# Qwen3.8-27B Uncensored (HauhauCS “Aggressive” · MTP) — Colab chat v1: **IQ2_M** • max context • max reasoning • tool calls • web search • public link

**How to use**
1. **Runtime → Change runtime type → T4 GPU** (preselected if you opened this file in Colab).
2. **Cell 1** — pick the model file (default **IQ2_M**; every quant of the repo is in the list) and run it. It mounts Google Drive first (optional build cache), then builds llama.cpp and downloads the pre-quantized GGUF **plus the vision encoder** (`mmproj`, ~0.9 GB) *in parallel* (~10–20 min the first time; re-running is fast). No conversion needed. Optional: choose the **FastMTP** build there for the fastest speed-up.
3. **Cell 2** — keep the defaults (**Max** context, **Max** reasoning, **Parallel** web search) or change them, then run it. It shows the chat right in the notebook, plus a **public link**.
4. **Keep this tab open and leave Cell 2 running.** It watches the model server and the link and repairs them automatically (details below).

**What this model is (and how the notebook handles it)**
- **A dense 27B, hybrid architecture** (`qwen35` family — the same family as the 9B notebook, just bigger): 64 layers, of which **only 16 keep a KV cache**; the other 48 are Gated-DeltaNet layers with a small fixed-size state. That is why very long windows are possible at all. Cell 2 reads the real layer counts from the GGUF file itself.
- **IQ2_M is the practical choice for a 15 GB T4.** The file is 10.3 GB, which leaves room for a very long chat memory. Bigger quants are listed with a note on what GPU they need; a file that is too big for the GPU runs partly on the CPU (slow, 1–3 tokens/s) instead of failing.
- **Context: as large as the GPU can hold, up to 262,144 (the native limit).** The memory planner picks the *biggest* window that fits and, at that window, the best chat-memory precision (16-bit → 8-bit → 4-bit). **Estimate for a T4 + IQ2_M** (image encoder off, MTP off): about **180K–196K tokens** with 4-bit chat memory. The full 262,144 needs a bigger GPU (on an A100 40 GB the planner reaches it by itself). The exact number is printed by Cell 2.
- **What eats context** (each is a switch in Cell 2, so you decide): the **image encoder on the GPU** costs ~1.5 GB (≈ −80K tokens), **embedded MTP** about −25 %, **FastMTP** about −40 %. That is why **vision and MTP are OFF by default** — you asked for the maximum window. Everything is a dropdown; nothing needs code changes.
- **Max reasoning.** The model thinks natively inside `<think>…</think>` (a collapsible panel in the chat). **Max** = the model's **`xhigh`** reasoning effort with **no limit** on the thinking length (default). **High / Medium / Low** add a hard thinking-token budget enforced by llama-server; **Off** = no thinking. Sampling follows the model card: thinking → temperature **1.0**, top-p 0.95, top-k 20, no repetition penalty; non-thinking → 0.7 / 0.8 / 20 with presence penalty 1.5.
- **MTP (multi-token prediction) = same answers, faster.** The model drafts several tokens ahead and verifies them. *Embedded MTP* uses the head that is inside the model file (works with the standard build). *FastMTP* uses the author's 0.9 GB draft file and needs his small llama.cpp patch — Cell 1 applies it for you (to the verified release if it fits, otherwise to the author's exact commit) and tells you what happened. If a build rejects MTP — for example together with image input — Cell 2 switches it off by itself and keeps everything else.
- **The model's own chat template is used unchanged** (native thinking, preserved reasoning, XML-style tool calls with llama.cpp's matching parser).
- **Vision (optional):** switch it on in Cell 2 and attach images with the **+** button (PNG/JPEG). Cell 2 tests it at start-up with a red square; if that fails it moves the encoder to the CPU or turns vision off and tells you. Video is not supported by the llama.cpp chat UI — send frames.
- **Tools & web:** the model gets `web_search` and `web_fetch` (Parallel + Exa + DuckDuckGo with auto-fallback; the keys are pre-filled, and the Parallel key also adds the slow `web_research` tool). Results are capped relative to the context size and every call is time-bounded. Use the **Allow tool calls: ON / OFF** switch inside the chat box, right after the + button.
- **No context shifting:** hybrid models cannot drop their oldest turns. If a chat truly fills the window it stops with a “context full” message — start a new chat. A single reply is capped automatically at 70 % of the window (at most 131,072 tokens) so one runaway reply can never use up the whole window; set `MAX_TOKENS = -1` in the advanced block for no cap.

**Built for long sessions** (hours of chatting and work)
- The watchdog restarts a **crashed** or **truly hung** server by itself — but **never** a server that is only busy: a very long prompt can keep a T4 silent for minutes, and while the GPU is computing it is left alone. After a restart it waits for the GPU memory to be released and plans again, so the window does not shrink.
- Restarts are only counted for 30 minutes, so a session running for days never runs out of them. If a restart ever needs another port, the public link is re-pointed automatically.
- Host RAM is protected (prompt cache sized to the machine, a fixed number of saved snapshots), only the end of the server log is ever read, and every 10 minutes Cell 2 prints GPU memory, free RAM and free disk.
- Requests may take up to 2 hours; the connection is kept alive while the model thinks.

**About the model:** Qwen3.8-27B with refusals removed by HauhauCS (“Aggressive” variant). It may still append a short disclaimer — that comes from the base model's training. Use responsibly and only where that is allowed.
- **Public link:** a **Cloudflare** link (no account, new address each run). The chat inside the notebook always works too.
- **Long input:** 1,000+ lines of text or code is fine — paste it or attach the file. (On a T4 a very long prompt takes minutes to read; that is normal.)
- Chats are stored in your browser tab. If the public link changes, it's a new origin, so export chats you care about from the UI settings.

In [ ]:
#@title 1️⃣ Setup — build llama.cpp and download Qwen3.8-27B (IQ2_M default + vision encoder, ≈10–20 min first run) { display-mode: "form" }
MODEL_QUANT = "IQ2_M · 10.3 GB · 3.02 bpw · T4: the longest context (default)" #@param ["IQ2_M · 10.3 GB · 3.02 bpw · T4: the longest context (default)", "Q2_K_P · 10.7 GB · 3.12 bpw · T4: long context", "IQ3_XS · 12.2 GB · 3.56 bpw · T4: roughly 90K context", "IQ3_M · 12.8 GB · 3.74 bpw · T4: roughly 55K context", "Q3_K_P · 13.4 GB · 3.93 bpw · T4: roughly 25K context", "IQ4_XS · 15.7 GB · 4.60 bpw · needs a bigger GPU (L4 / A100 …)", "Q4_K_P · 17.9 GB · 5.25 bpw · needs a bigger GPU (L4 / A100 …)", "Q5_K_P · 20.2 GB · 5.92 bpw · needs a 24 GB+ GPU", "Q6_K_P · 25.9 GB · 7.59 bpw · needs a 40 GB GPU", "Q8_K_P · 31.5 GB · 9.21 bpw · needs a 40 GB GPU"]
DOWNLOAD_VISION = True #@param {type:"boolean"}
LLAMA_BUILD = "Standard · llama.cpp b11925, built-in MTP available (default)" #@param ["Standard · llama.cpp b11925, built-in MTP available (default)", "FastMTP · patched build + the 0.9 GB FastMTP draft file (fastest, a little less context)"]
USE_DRIVE_CACHE = True #@param {type:"boolean"}
# MODEL_QUANT: the repo ships pre-quantized GGUFs — this is a plain download, no conversion. EVERY quant of the repo is listed.
#   IQ2_M is the primary target: it is the only 27B file that leaves a free 15 GB T4 room for a very long context
#   (the model keeps a KV cache in only 16 of its 64 layers). Bigger files fit only on bigger GPUs; if a file is too big for the GPU,
#   Cell 2 runs it partly on the CPU (slow) instead of failing. The labels say what a T4 can expect.
#   If the chosen quant is missing from the repo, the next smaller one is picked automatically.
# DOWNLOAD_VISION: also downloads the ~0.9 GB vision encoder (mmproj) so Cell 2 can switch image input on (it is OFF by default there,
#   because it costs GPU memory that is otherwise context).
# LLAMA_BUILD: "Standard" is the llama.cpp release this notebook family was verified with (chat UI + web tools). It already has MTP
#   (multi-token-prediction) speed-up built into the model file. "FastMTP" additionally applies the model author's patch and downloads his
#   0.9 GB draft file, for a bigger speed-up. If the patch does not apply it falls back to the author's exact commit, then to Standard.
# USE_DRIVE_CACHE: mounts Google Drive first and saves the compiled server there so the next session skips the ~15 min build.

import os, sys, re, time, json, shutil, hashlib, subprocess
from concurrent.futures import ThreadPoolExecutor

SCRATCH = "/content"
QUANT     = MODEL_QUANT.split()[0]                      # e.g. "IQ2_M"
REPO_ID   = "HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF"
NAME_BASE = "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive"
FILENAME  = f"{NAME_BASE}-{QUANT}.gguf"
MMPROJ_FILE = f"mmproj-{NAME_BASE}-BF16.gguf"           # the vision encoder
DRAFT_FILE  = f"{NAME_BASE}-FastMTP-32K.gguf"           # the FastMTP draft sidecar
DRAFT_SHA256 = "115e618e1f73cb50817ed5856f0551c6bf9c3d94df96f440eaca78dc63b8968b"   # from the model card
PATCH_FILE  = "HauhauCS-FastMTP-llama.cpp.patch"
PREFERENCE = ["Q8_K_P", "Q6_K_P", "Q5_K_P", "Q4_K_P", "IQ4_XS", "Q3_K_P", "IQ3_M", "IQ3_XS", "Q2_K_P", "IQ2_M"]   # fallback order (best -> smallest)
SIZE_GB   = {"Q8_K_P": 31.46, "Q6_K_P": 25.92, "Q5_K_P": 20.22, "Q4_K_P": 17.92, "IQ4_XS": 15.71,
             "Q3_K_P": 13.44, "IQ3_M": 12.79, "IQ3_XS": 12.18, "Q2_K_P": 10.68, "IQ2_M": 10.32}   # for the progress display only
LLAMA_TAG = "b11925"                                    # pinned llama.cpp release with a matching chat UI (the one this notebook family was verified with)
FASTMTP_COMMIT = "4df29be4f4c3673f428170fda944a5b19f743bb8"   # the exact llama.cpp commit the model card builds FastMTP on
WANT_FASTMTP = LLAMA_BUILD.startswith("FastMTP")
LLAMA_URL = "https://github.com/ggml-org/llama.cpp"

LLAMA_DIR = "/content/llama.cpp"
BIN_PATH  = f"{LLAMA_DIR}/build/bin/llama-server"
STAMP     = f"{LLAMA_DIR}/build/qwen38_build_stamp.txt"
MODEL_DIR = "/content/models"
CF_PATH   = "/content/cloudflared"
CONFIG    = "/content/qwen38_27b_config.json"
BUILD_LOG = "/content/build.log"
CF_LOG    = "/content/cf.log"
UI_TGZ    = "/content/llama-ui.tar.gz"
WEBUI_SRC = "/content/webui_src"
CACHE_DIR = "/content/drive/MyDrive/qwen38_cache"
os.environ["PATH"] += ":/usr/local/cuda/bin"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.makedirs(MODEL_DIR, exist_ok=True)

# ==== PURE-BEGIN  (no side effects: unit-tested outside Colab) ====
import os, re, struct, hashlib

# ---------------------------------------------------------------- web UI tarball (any archive layout)
def extract_ui(tgz, dest):
    """Unpack llama-<tag>-ui.tar.gz into `dest`, whatever the top-level layout is: the directory that
    holds index.html (shallowest one) becomes `dest`."""
    import tarfile, shutil
    tmp = dest + ".tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp)
    with tarfile.open(tgz) as t:
        try:
            t.extractall(tmp, filter="data")
        except TypeError:                       # Python without extraction filters
            t.extractall(tmp)
    root, best = None, 10**6
    for r, _dirs, files in os.walk(tmp):
        if "index.html" in files:
            depth = os.path.relpath(r, tmp).count(os.sep)
            if depth < best:
                best, root = depth, r
    if root is None:
        shutil.rmtree(tmp, ignore_errors=True)
        raise RuntimeError("the UI archive has no index.html")
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(root, dest)
    shutil.rmtree(tmp, ignore_errors=True)
    return dest


# ---------------------------------------------------------------- choose a GGUF file in a repo listing
def pick_gguf(files, quant, preference, exact_name=None):
    """-> (filename, quant_used) or None. Exact name first, then the requested quant, then the next smaller one.
    Never returns the vision projector or the FastMTP draft sidecar."""
    ggufs = sorted(f for f in files if f.lower().endswith(".gguf") and "mmproj" not in f.lower()
                   and "fastmtp" not in f.lower() and "/" not in f and not re.search(r"-\d{5}-of-\d{5}", f))
    if exact_name and exact_name in ggufs:
        return exact_name, quant
    order = preference[preference.index(quant):] if quant in preference else list(preference)
    for q in order:
        pat = re.compile(r"(?<![A-Za-z0-9])" + re.escape(q) + r"(?![A-Za-z0-9])", re.I)
        hit = [f for f in ggufs if pat.search(f)]
        if hit:
            return hit[0], q
    return None


def pick_mmproj(files, preferred=""):
    """Vision-encoder GGUF in a repo listing: the preferred name, else BF16 > F16 > F32 > anything called mmproj*."""
    cands = sorted(f for f in files if f.lower().startswith("mmproj") and f.lower().endswith(".gguf") and "/" not in f)
    if preferred and preferred in cands:
        return preferred
    for tag in ("bf16", "f16", "f32"):
        for f in cands:
            if tag in f.lower():
                return f
    return cands[0] if cands else ""


def pick_draft(files, preferred=""):
    """The FastMTP draft sidecar in a repo listing ('' if the repo has none)."""
    cands = sorted(f for f in files if "fastmtp" in f.lower() and f.lower().endswith(".gguf") and "/" not in f)
    if preferred and preferred in cands:
        return preferred
    return cands[0] if cands else ""


def is_sha(ref):
    """True for a full 40-character git commit hash (otherwise `ref` is a tag)."""
    return bool(re.fullmatch(r"[0-9a-f]{40}", ref or ""))


def build_variant(ref, patched):
    """Short name of one llama.cpp build: 'b11925', 'b11925-fastmtp', '4df29be4f4c3-fastmtp'."""
    return (ref[:12] if is_sha(ref) else ref) + ("-fastmtp" if patched else "")


def build_stamp(ref, patched, arch, fallback=False):
    """Content of the stamp file next to a compiled server. `fallback` = the FastMTP patch could not be applied anywhere."""
    return f"{build_variant(ref, patched)} sm{arch}" + (" fastmtp-unavailable" if fallback else "")


def build_plan(want_fastmtp, tag, commit):
    """-> (accepted, compile_order). Each item is (ref, patched, fallback).
    accepted      : builds already on disk / in the Drive cache that satisfy the request (checked in this order)
    compile_order : what to compile when nothing usable exists (first success wins)."""
    if want_fastmtp:
        order = [(tag, True, False), (commit, True, False), (tag, False, True)]
        return list(order), list(order)
    return [(tag, False, False), (tag, True, False), (commit, True, False), (tag, False, True)], [(tag, False, False)]


def sha256_file(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()
# ==== PURE-END ====


# ======================= shared helpers (proven in the earlier notebooks of this family) =======================
import threading
SRC_LOCK = threading.Lock()

def run(cmd, log, env=None, retries=1):
    for attempt in range(retries):
        with open(log, "a") as f:
            f.write(f"\n$ {' '.join(cmd)}\n"); f.flush()
            p = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env)
        if p.returncode == 0:
            return
        if attempt + 1 < retries:
            time.sleep(5)
    tail = open(log, errors="replace").read()[-4000:]
    raise RuntimeError(f"Command failed: {' '.join(cmd)}\n\n{tail}")

def binary_ok(path):
    try:
        out = subprocess.run([path, "--version"], capture_output=True, text=True, timeout=60)
        return out.returncode == 0
    except Exception:
        return False

def find_libcuda():
    """Path of the CUDA driver library (libcuda.so / libcuda.so.1), or '' if absent."""
    paths = []
    try:
        out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout or ""
        for line in out.splitlines():
            if "libcuda.so" in line and "=>" in line:
                paths.append(line.rsplit("=> ", 1)[-1].strip())
    except Exception:
        pass
    paths += ["/usr/lib/x86_64-linux-gnu/libcuda.so.1", "/usr/lib64/libcuda.so.1",
              "/usr/local/cuda/compat/libcuda.so.1",
              "/usr/local/cuda/lib64/stubs/libcuda.so",
              "/usr/local/cuda/targets/x86_64-linux/lib/stubs/libcuda.so"]
    for p in paths:
        if p and os.path.exists(p):
            return p
    return ""

def ensure_cuda_driver_link():
    """CMake's FindCUDAToolkit sometimes cannot see the driver library (CUDA::cuda_driver); a libcuda.so
    symlink next to the toolkit libs fixes that. No-op when CMake can already find it."""
    need = [d for d in ("/usr/local/cuda/lib64", "/usr/local/cuda/targets/x86_64-linux/lib")
            if os.path.isdir(d) and not os.path.exists(f"{d}/libcuda.so")]
    if not need:
        return ""
    drv = find_libcuda()
    if not drv:
        return ""
    for d in need:
        try:
            os.symlink(drv, f"{d}/libcuda.so")
            print(f"🔧 linked {drv} → {d}/libcuda.so (CUDA::cuda_driver fix)")
        except OSError:
            pass
    return drv

def prepare_webui():
    """Download + unpack the official prebuilt chat UI of this exact release (no Node.js needed).
    Cell 2 patches a copy of it and serves it with --path, so the server build never depends on it."""
    if os.path.isfile(f"{WEBUI_SRC}/index.html"):
        return "already there"
    url = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}/llama-{LLAMA_TAG}-ui.tar.gz"
    run(["wget", "-q", "--tries=3", "-O", UI_TGZ, url], BUILD_LOG)
    extract_ui(UI_TGZ, WEBUI_SRC)
    return "downloaded"

def get_cloudflared():
    if os.path.exists(CF_PATH) and os.path.getsize(CF_PATH) > 1_000_000 and binary_ok(CF_PATH):
        return "already there"
    run(["wget", "-q", "--tries=3", "-O", CF_PATH,
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        CF_LOG, retries=3)
    os.chmod(CF_PATH, 0o755)
    if os.path.getsize(CF_PATH) < 1_000_000 or not binary_ok(CF_PATH):
        os.remove(CF_PATH)
        raise RuntimeError("cloudflared download looks broken; run this cell again.")
    return "downloaded"

PIP_LOCK = threading.Lock()      # two pip processes at once can corrupt each other

def pip_install(pkgs):
    with PIP_LOCK:
        p = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input"] + pkgs,
                           capture_output=True, text=True)
    return p.returncode == 0, (p.stderr or p.stdout or "").strip()[-200:]

def get_web_deps():
    # DuckDuckGo search + page-to-text: used by the DuckDuckGo option and as automatic fallback
    ok, err = pip_install(["ddgs", "trafilatura"])
    return "installed" if ok else "skipped (pip failed; Parallel/Exa still work)"

def hf_token():
    try:
        from google.colab import userdata
        return (userdata.get("HF_TOKEN") or "").strip()
    except Exception:
        return ""

class Heartbeat(threading.Thread):
    """Prints a progress line every few seconds while a download/conversion runs."""
    def __init__(self, label, watch, is_dir, interval=10, total_bytes=0):
        super().__init__(daemon=True)
        self.label, self.watch, self.is_dir = label, watch, is_dir
        self.interval, self.total = interval, total_bytes
        self.t0 = time.time()
        self.stop_event = threading.Event()

    def _size(self):
        try:
            if self.is_dir:
                total = 0
                for root, _, files in os.walk(self.watch):
                    for name in files:
                        try:
                            total += os.path.getsize(os.path.join(root, name))
                        except OSError:
                            pass
                return total
            return os.path.getsize(self.watch) if os.path.exists(self.watch) else 0
        except OSError:
            return 0

    def run(self):
        while not self.stop_event.wait(self.interval):
            cur = self._size()
            el = int(time.time() - self.t0)
            msg = f"   [{self.label}] {el // 60}:{el % 60:02d} | {cur/1e9:5.1f} GB"
            if self.total:
                msg += f" / ~{self.total/1e9:.0f} GB"
            try:
                msg += f" | {shutil.disk_usage(SCRATCH).free/1e9:5.1f} GB free"
            except OSError:
                pass
            print(msg, flush=True)

    def stop(self):
        self.stop_event.set()
        self.join(timeout=5)

def gpu_check():
    try:
        gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                                       "--format=csv,noheader"], text=True).strip().splitlines()[0]
        name, cap, vram = [x.strip() for x in gpu.split(",")]
        print(f"✅ GPU: {name} (sm_{cap.replace('.', '')}, {vram})")
        return cap.replace(".", "")
    except Exception:
        raise RuntimeError("❌ No GPU found. Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")

def mount_drive():
    global USE_DRIVE_CACHE
    if USE_DRIVE_CACHE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            print("✅ Google Drive mounted (build cache)")
        except Exception as e:
            USE_DRIVE_CACHE = False
            print(f"⚠️ Could not mount Google Drive ({e}); continuing without the build cache.")
    return USE_DRIVE_CACHE

def wait_jobs(jobs, t0):
    last = -1
    while not all(j.done() for j in jobs.values()):
        time.sleep(5)
        mins = int((time.time() - t0) // 60)
        if mins != last:
            last = mins
            print(f"   {mins:>2} min | " + " | ".join(f"{k}: {'done' if j.done() else '…'}" for k, j in jobs.items()))
    errors = [f"[{k}] {j.exception()}" for k, j in jobs.items()
              if j.exception() is not None and k != "web tools"]
    if errors:
        raise RuntimeError("Setup failed — just run this cell again (finished parts are reused).\n\n" + "\n\n".join(errors))



# ======================= Qwen3.8-27B specific: llama.cpp source, optional FastMTP patch, build =======================
def ensure_llama_source(ref, patch=None):
    """Checkout of `ref` (a release tag or a full commit hash), optionally with the FastMTP patch applied.
    Thread-safe and idempotent. Raises RuntimeError if the patch does not apply to this checkout."""
    with SRC_LOCK:
        marker = f"{LLAMA_DIR}/.pinned_ref"
        want = f"{ref}|{'patch' if patch else 'plain'}"
        if (os.path.exists(f"{LLAMA_DIR}/CMakeLists.txt") and os.path.exists(marker)
                and open(marker).read().strip() == want):
            return "already there"
        shutil.rmtree(LLAMA_DIR, ignore_errors=True)
        open(BUILD_LOG, "a").close()
        if is_sha(ref):                              # a commit, not a tag: fetch exactly that one commit
            os.makedirs(LLAMA_DIR)
            run(["git", "init", "-q", LLAMA_DIR], BUILD_LOG)
            run(["git", "-C", LLAMA_DIR, "remote", "add", "origin", LLAMA_URL], BUILD_LOG)
            run(["git", "-C", LLAMA_DIR, "fetch", "--depth", "1", "origin", ref], BUILD_LOG, retries=3)
            run(["git", "-C", LLAMA_DIR, "-c", "advice.detachedHead=false", "checkout", "-q", "FETCH_HEAD"], BUILD_LOG)
        else:
            run(["git", "-c", "advice.detachedHead=false", "clone", "--depth", "1", "--branch", ref, LLAMA_URL, LLAMA_DIR], BUILD_LOG, retries=3)
        if patch:
            run(["git", "-C", LLAMA_DIR, "apply", "--check", patch], BUILD_LOG)     # raises if it does not apply
            run(["git", "-C", LLAMA_DIR, "apply", patch], BUILD_LOG)
        open(marker, "w").write(want)
        return "cloned" + (" + FastMTP patch" if patch else "")

def get_hf_file(name):
    """One small/medium file of the model repo (patch, draft sidecar) into MODEL_DIR."""
    try:
        import huggingface_hub  # noqa
    except ImportError:
        pip_install(["huggingface_hub"])
    from huggingface_hub import hf_hub_download
    last = None
    for attempt in range(3):
        try:
            return hf_hub_download(REPO_ID, name, local_dir=MODEL_DIR, token=hf_token() or None)
        except Exception as e:                      # noqa: BLE001
            last = e
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"could not download {name}: {last}")

def short_error(e):
    """The meaningful part of a failed command's output: its 'error:'/'fatal:' lines (else the last line)."""
    lines = [l.strip() for l in str(e).splitlines() if l.strip()]
    hits = [l for l in lines if re.search(r"\b(error|fatal)\b", l, re.I)]
    return " / ".join((hits or lines[-1:])[-2:])[:200]

def cache_file(ref, patched, fallback):
    return f"{CACHE_DIR}/llama-server-{build_variant(ref, patched)}-sm{ARCH}" + ("-fallback" if fallback else "")

def compile_server(ref, patched, patch_path):
    """Checkout (+patch) and compile llama-server. Raises RuntimeError on any failure."""
    ensure_llama_source(ref, patch_path if patched else None)
    env = dict(os.environ)
    drv = ensure_cuda_driver_link()
    # -DLLAMA_BUILD_UI=OFF: the chat UI is served from files (--path), so a UI download/embedding problem
    # can never break the compile.
    cmake_cmd = ["cmake", "-S", LLAMA_DIR, "-B", f"{LLAMA_DIR}/build",
                 "-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={ARCH}",
                 "-DBUILD_SHARED_LIBS=OFF", "-DLLAMA_OPENSSL=OFF", "-DLLAMA_BUILD_UI=OFF",
                 "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF",
                 "-DCMAKE_BUILD_TYPE=Release"]
    try:
        run(cmake_cmd, BUILD_LOG, env=env)
    except RuntimeError as e:
        if not (drv and "cuda_driver" in str(e)):
            raise
        run(cmake_cmd + [f"-DCUDA_DRIVER_LIBRARY={drv}"], BUILD_LOG, env=env)
    run(["cmake", "--build", f"{LLAMA_DIR}/build", "--config", "Release", "--target", "llama-server",
         "-j", str(min(os.cpu_count() or 2, 4))], BUILD_LOG, env=env)
    if not binary_ok(BIN_PATH):
        raise RuntimeError(f"The server was built but does not start. See {BUILD_LOG}")

def build_server():
    """-> (status text, info dict {ref, patched, fallback}). Reuses a build that is already there, then the Drive cache,
    then compiles. With the FastMTP option the patched variants are tried first; the last resort is a plain build."""
    accepted, order = build_plan(WANT_FASTMTP, LLAMA_TAG, FASTMTP_COMMIT)
    def info(item):
        return {"ref": item[0], "patched": item[1], "fallback": item[2]}
    good = os.path.exists(BIN_PATH) and binary_ok(BIN_PATH)
    if good and os.path.exists(STAMP):
        have = open(STAMP).read().strip()
        for item in accepted:
            if have == build_stamp(item[0], item[1], ARCH, item[2]):
                return "already built", info(item)
    if USE_CACHE:
        os.makedirs(os.path.dirname(BIN_PATH), exist_ok=True)
        for item in accepted:
            cf = cache_file(*item)
            if os.path.exists(cf):
                shutil.copy(cf, BIN_PATH); os.chmod(BIN_PATH, 0o755)
                if binary_ok(BIN_PATH):
                    open(STAMP, "w").write(build_stamp(item[0], item[1], ARCH, item[2]))
                    return "restored from Drive cache", info(item)
    patch_path, errors = None, []
    for item in order:
        ref, patched, fallback = item
        if patched and patch_path is None:
            try:
                patch_path = get_hf_file(PATCH_FILE)
            except RuntimeError as e:
                errors.append(str(e)); patch_path = ""
        if patched and not patch_path:
            continue
        try:
            compile_server(ref, patched, patch_path)
        except RuntimeError as e:
            errors.append(f"{build_variant(ref, patched)}: {short_error(e)}")
            if not patched:
                raise
            continue
        os.makedirs(os.path.dirname(STAMP), exist_ok=True)
        open(STAMP, "w").write(build_stamp(ref, patched, ARCH, fallback))
        if USE_CACHE:
            try:
                shutil.copy(BIN_PATH, cache_file(*item))
            except OSError as e:
                print(f"⚠️ could not save the build to Drive ({e}) - continuing")
        why = ("" if not errors else " (earlier attempts failed: " + "; ".join(x[:200] for x in errors) + ")")
        return "built from source" + why, info(item)
    raise RuntimeError("Could not build the server:\n" + "\n".join(errors))

def build_job():
    try:
        return build_server()
    except BaseException as e:      # noqa: BLE001
        raise

def download_mmproj(files, token):
    """The ~0.9 GB vision encoder. Never fatal: without it the model simply runs text-only."""
    if not DOWNLOAD_VISION:
        return ""
    from huggingface_hub import hf_hub_download
    name = pick_mmproj(files, MMPROJ_FILE) if files else MMPROJ_FILE
    if not name:
        print("⚠️ No vision encoder (mmproj) file found in the repo - continuing text-only.")
        return ""
    for attempt in range(3):
        try:
            return hf_hub_download(REPO_ID, name, local_dir=MODEL_DIR, token=token)
        except Exception as e:                  # noqa: BLE001
            print(f"⚠️ vision encoder download attempt {attempt + 1}/3 failed: {str(e)[:160]}")
            time.sleep(10 * (attempt + 1))
    print("⚠️ Could not download the vision encoder - continuing text-only (re-run Cell 1 to retry).")
    return ""

def download_draft(files, token):
    """The ~0.9 GB FastMTP draft file (only for the FastMTP build). Never fatal."""
    if not WANT_FASTMTP:
        return ""
    from huggingface_hub import hf_hub_download
    name = pick_draft(files, DRAFT_FILE) if files else DRAFT_FILE
    if not name:
        print("⚠️ No FastMTP draft file in the repo - FastMTP will not be available.")
        return ""
    for attempt in range(3):
        try:
            path = hf_hub_download(REPO_ID, name, local_dir=MODEL_DIR, token=token)
            if os.path.basename(path) == DRAFT_FILE and sha256_file(path) != DRAFT_SHA256:
                print("⚠️ the FastMTP draft file's SHA-256 differs from the model card (the author may have re-published it) - using it anyway.")
            return path
        except Exception as e:                  # noqa: BLE001
            print(f"⚠️ FastMTP draft download attempt {attempt + 1}/3 failed: {str(e)[:160]}")
            time.sleep(10 * (attempt + 1))
    print("⚠️ Could not download the FastMTP draft - FastMTP will not be available (re-run Cell 1 to retry).")
    return ""

def download_model():
    try:
        import huggingface_hub  # noqa
    except ImportError:
        pip_install(["huggingface_hub"])
    from huggingface_hub import hf_hub_download, list_repo_files
    token = hf_token() or None
    fname, quant_used = FILENAME, QUANT
    files = []
    try:
        files = list_repo_files(REPO_ID, token=token)
        picked = pick_gguf(files, QUANT, PREFERENCE, exact_name=FILENAME)
        if not picked:
            raise RuntimeError(f"No usable GGUF quant found in {REPO_ID}. Files:\n" + "\n".join(files))
        fname, quant_used = picked
        if fname != FILENAME:
            print(f"   (no {QUANT} file in the repo — using {fname})")
    except RuntimeError:
        raise
    except Exception as e:                      # noqa: BLE001
        print(f"   (could not list repo files: {e}; trying the standard file name)")
    keep = (os.path.basename(fname),)
    for old in os.listdir(MODEL_DIR):           # free disk: other quants from earlier runs
        if old.endswith(".gguf") and old not in keep and not old.startswith("mmproj") and "FastMTP" not in old:
            try:
                os.remove(os.path.join(MODEL_DIR, old))
            except OSError:
                pass
    last = None
    for attempt in range(4):
        try:
            path = hf_hub_download(REPO_ID, fname, local_dir=MODEL_DIR, token=token)
            return path, quant_used, download_mmproj(files, token), download_draft(files, token)
        except Exception as e:                  # noqa: BLE001
            last = e
            print(f"⚠️ download attempt {attempt + 1}/4 failed: {str(e)[:200]}")
            time.sleep(10 * (attempt + 1))
    raise RuntimeError(f"Download of {fname} failed after 4 tries: {last}")

def download_job():
    total = (SIZE_GB.get(QUANT, 10.3) + (0.93 if DOWNLOAD_VISION else 0) + (0.9 if WANT_FASTMTP else 0)) * 1e9
    hb = Heartbeat("download", MODEL_DIR, True, 15, int(total))
    hb.start()
    try:
        return download_model()
    finally:
        hb.stop()

def main():
    global ARCH, USE_CACHE
    mount_drive()
    USE_CACHE = USE_DRIVE_CACHE
    ARCH = gpu_check()
    need_gb = SIZE_GB.get(QUANT, 10.3) + 8
    free_gb = shutil.disk_usage(SCRATCH).free / 1e9
    if free_gb < need_gb + 6:
        print(f"⚠️ Only {free_gb:.0f} GB of disk free; the build + this model need ~{need_gb + 6:.0f} GB.")
    if USE_CACHE:
        os.makedirs(CACHE_DIR, exist_ok=True)

    t0 = time.time()
    print(f"⏳ Building llama.cpp ({'FastMTP-capable' if WANT_FASTMTP else LLAMA_TAG}) and downloading {REPO_ID} ({QUANT}) at the same time...")
    with ThreadPoolExecutor(max_workers=4) as ex:
        jobs = {"build": ex.submit(build_job), "download": ex.submit(download_job),
                "tunnel tool": ex.submit(get_cloudflared), "web tools": ex.submit(get_web_deps)}
        wait_jobs(jobs, t0)
    build_status, binfo = jobs["build"].result()
    model_path, quant_used, mmproj_path, draft_path = jobs["download"].result()
    cf_status = jobs["tunnel tool"].result()
    web_status = jobs["web tools"].result()
    try:
        ui_status = prepare_webui()
    except Exception as e:                      # noqa: BLE001
        ui_status = f"will be fetched by Cell 2 ({str(e)[:80]})"

    cfg = {"model": model_path, "quant": quant_used, "bin": BIN_PATH, "cloudflared": CF_PATH,
           "tag": LLAMA_TAG, "ui_tag": LLAMA_TAG, "webui_src": WEBUI_SRC, "mmproj": mmproj_path,
           "draft": draft_path, "fastmtp_patched": bool(binfo["patched"]), "llama_ref": binfo["ref"]}
    json.dump(cfg, open(CONFIG, "w"))
    which = ("FastMTP patch applied" if binfo["patched"] else "standard") + f", llama.cpp {build_variant(binfo['ref'], False)}"
    print(f"\n✅ Server:      {build_status} ({which})")
    if WANT_FASTMTP and not binfo["patched"]:
        print("⚠️ FastMTP:    the patch could not be applied to any llama.cpp version tried (see /content/build.log). "
              "Cell 2 will offer the built-in MTP instead.")
    print(f"✅ Model:       {os.path.basename(model_path)} ({os.path.getsize(model_path)/1e9:.1f} GB, {quant_used})")
    print("✅ Vision:      " + (f"{os.path.basename(mmproj_path)} ({os.path.getsize(mmproj_path)/1e9:.2f} GB) — switch it on in Cell 2 if you need images" if mmproj_path else "not downloaded (text-only)"))
    if WANT_FASTMTP:
        print("✅ FastMTP:     " + (f"{os.path.basename(draft_path)} ({os.path.getsize(draft_path)/1e9:.2f} GB)" if draft_path else "draft file missing"))
    print(f"✅ Tunnel tool: {cf_status}")
    print(f"✅ Web tools:   {web_status}")
    print(f"✅ Chat UI:     {ui_status}")
    print(f"⏱  {int(time.time() - t0)//60} min {int(time.time() - t0)%60} s.  👉 Now run Cell 2.")

if __name__ == "__main__":
    main()


In [ ]:
#@title 2️⃣ Launch — choose context, reasoning and web search, then run (keep this cell running!) { display-mode: "form" }
VISION_MODE = "Off · keeps the GPU free for context (default)" #@param ["Off · keeps the GPU free for context (default)", "On · image encoder on the CPU (slow, costs no GPU memory)", "On · image encoder on the GPU (fast, costs ~1.5 GB of context room)"]
CONTEXT_WINDOW = "Max · the largest window this GPU can hold (default)" #@param ["Max · the largest window this GPU can hold (default)", "262144 · the native limit", "229376", "196608", "163840", "131072", "98304", "65536", "32768", "16384", "8192"]
KV_CACHE = "auto · best precision that fits the chosen context (default)" #@param ["auto · best precision that fits the chosen context (default)", "f16 · full precision, less context", "q8_0 · 8-bit: higher quality, less context", "q4_0 · 4-bit: the most context"]
REASONING = "Max · xhigh effort, unlimited thinking (default)" #@param ["Max · xhigh effort, unlimited thinking (default)", "High · xhigh effort, up to 16384 thinking tokens", "Medium · medium effort, up to 4096 thinking tokens", "Low · low effort, up to 1024 thinking tokens", "Off · answers directly, no thinking"]
SPECULATIVE = "Off · maximum context (default)" #@param ["Off · maximum context (default)", "Embedded MTP · about 1.6–2.2× faster, uses some context", "FastMTP · fastest (about 1.9–3×), needs the FastMTP build from Cell 1, uses the most context"]
WEB_SEARCH = "Parallel · free, no key needed (default)" #@param ["Parallel · free, no key needed (default)", "Exa · free tier is rate-limited, best with a key", "DuckDuckGo · free, no key needed", "Off · no web tools"]
AUTO_FALLBACK = True #@param {type:"boolean"}
PARALLEL_API_KEY = "VtYAbqEv6_sOvxUtcGeHI28jHmIpGuxM_talrLr6" #@param {type:"string"}
EXA_API_KEY = "29334ebe-08ca-4c98-8b57-af84986d1464" #@param {type:"string"}
PUBLIC_LINK = "Cloudflare · no account, new link each run (default)" #@param ["Cloudflare · no account, new link each run (default)", "Off · private Colab link only"]
SHOW_INLINE = True #@param {type:"boolean"}
# CONTEXT_WINDOW: "Max" = the biggest window that physically fits this GPU. The chat memory (KV cache) drops from 16-bit to 8-bit to
#   4-bit only as far as it takes. On a 15 GB T4 with IQ2_M the answer is roughly 180K-196K tokens (4-bit chat memory); the native
#   262,144 needs a bigger GPU. A fixed number is honoured the same way: the best memory precision that fits.
# VISION_MODE: image input. OFF by default because the image encoder on the GPU costs ~1.5 GB = roughly 80K tokens of context on a T4.
#   "CPU" keeps the encoder in system RAM (each image then takes tens of seconds). The encoder file must have been downloaded in Cell 1.
# KV_CACHE: the chat memory. 8-bit is almost lossless; 4-bit doubles the context but is a bit less precise.
# REASONING: Qwen3.8 thinks natively (<think> ... </think>, shown in a collapsible panel). Max = the model's "xhigh" reasoning effort with
#   NO limit on the thinking length. High/Medium/Low also cap the thinking tokens (enforced by llama-server). Off = no thinking.
# SPECULATIVE: MTP = the model drafts several tokens ahead and verifies them, so answers appear faster with identical text. It costs GPU memory
#   (a second small cache, and for FastMTP the 0.9 GB draft file), so the context window gets smaller: on a T4 about 25% (Embedded) to 40%
#   (FastMTP) less. Off keeps the biggest window. If the server rejects MTP (for example together with vision) it is switched off automatically.
# AUTO_FALLBACK: if the chosen search service is down or rate-limited, the other free ones are tried automatically.
# API keys are hardcoded above (source of truth). Colab secrets named PARALLEL_API_KEY / EXA_API_KEY are only
#   used if the field above is emptied. A Parallel key raises limits and adds a deep "research" tool.
# PUBLIC_LINK: anyone with the link can use your session. Cloudflare needs no account; the address changes each run.
# SHOW_INLINE: also shows the chat right here in the notebook (private, always works).

# ---------------- advanced (rarely needed) ----------------
MAX_TOKENS = None          # longest single reply (thinking + answer). None = automatic: 70% of the window, at most 131,072 - a runaway
                           # reply can then never use up the whole window; -1 = no cap at all; or a number
IMAGE_MAX_TOKENS = 1024    # vision: largest image the encoder will process (bigger = sharper but more memory)
PRESENCE_PENALTY = None    # None = Qwen's value for the mode (0 while thinking, 1.5 without thinking); set a number to override
PRESERVE_THINKING = True   # keep earlier turns' reasoning in the conversation (Qwen3.8's official default; keeps the prompt cache valid so
                           # follow-up turns start fast, but long chats fill the window sooner - set False to keep chats much shorter in tokens)
CACHE_RAM_MIB = None       # host-RAM prompt cache; None = automatic (about 12% of system RAM, 0.5-4 GB) so a 12 GB Colab never runs short
CTX_CHECKPOINTS = 8        # recurrent-state snapshots kept in system RAM (~0.15 GB each) so follow-up turns need no full re-read
REQUEST_TIMEOUT = 7200     # seconds a single request may take (very long prompts / thinking on a T4)
MONITOR_SECONDS = 20       # how often the watchdog looks at the server and the link
FORGET_RESTARTS_MIN = 30   # restarts older than this no longer count towards MAX_SERVER_RESTARTS
MTP_DEPTH = {"embedded": 2, "fast": 3}   # tokens drafted ahead (the model card's values)
BATCH, UBATCH = 2048, 512  # prompt-processing batch sizes (a smaller UBATCH saves ~0.5 GB of GPU memory but slows long prompts)
ALLOW_CPU_OFFLOAD = True   # a quant too big for the GPU runs partly on the CPU (slow, 1-3 tokens/s) instead of stopping
MIN_FREE_MIB     = 350     # GPU memory that must stay free after loading, so long chats never crash later
MAX_SERVER_RESTARTS = 5
# -----------------------------------------------------------
import os, re, sys, json, time, datetime, subprocess, shutil, socket
import requests
try:
    from IPython.display import display, HTML
except ImportError:                              # (only when run outside Colab/Jupyter)
    display = lambda *a, **k: None
    HTML = lambda s: s

PROFILE = dict(
    NAME="Qwen3.8-27B-Uncensored",
    N_LAYER=64, N_KV_HEAD=4, HEAD_DIM=256,           # hybrid: only 1 layer in 4 keeps a KV cache (all re-read from the GGUF at run time)
    FULL_ATTN_INTERVAL=4,                            # -> 16 attention layers x 4 KV heads x 256 dims (the other 48 are Gated-DeltaNet layers)
    CTX_LADDER=[262144, 229376, 196608, 180224, 163840, 147456, 131072, 114688, 98304, 81920, 65536, 49152, 32768, 24576, 16384, 8192, 4096],
    KV_AUTO_ORDER=["f16", "q8_0", "q4_0"],           # best precision first at a given context; Max still means the biggest window
    OVERHEAD_MIB=1500,                               # CUDA context + compute buffers (248K-token vocabulary) + the DeltaNet state (~0.2 GB)
    VISION_MIB=1500,                                 # image encoder weights (~0.9 GB) + its compute buffer
    SPEC_MIB=300,                                    # compute buffers of the MTP draft context (its cache + weights are added separately)
    CONTEXT_SHIFT=False,                             # hybrid/recurrent models cannot shift their context
    CONFIG="/content/qwen38_27b_config.json",
    IDENTITY="You are Qwen3.8-27B (uncensored, HauhauCS), a capable assistant that can use web tools and think carefully before answering.{vision} Today's date is {today}.",
)

# ==== PURE-BEGIN  (no side effects: unit-tested outside Colab) ====
import os, re, json, struct

# ---------------------------------------------------------------- minimal GGUF header reader
_SCALARS = {0: "<B", 1: "<b", 2: "<H", 3: "<h", 4: "<I", 5: "<i", 6: "<f", 7: "<?", 10: "<Q", 11: "<q", 12: "<d"}

def read_gguf_meta(path, want_prefixes=("general.", "tokenizer.chat_template"), buf_mb=96):
    """Return {key: value} for the small metadata keys (scalars + strings); big arrays are skipped.
    Pure python, no dependency. Raises ValueError if the file is not a GGUF."""
    with open(path, "rb") as f:
        buf = f.read(buf_mb << 20)
    if buf[:4] != b"GGUF":
        raise ValueError("not a GGUF file")
    ver = struct.unpack_from("<I", buf, 4)[0]
    if ver < 2:
        raise ValueError(f"unsupported GGUF version {ver}")
    n_kv = struct.unpack_from("<Q", buf, 16)[0]
    pos, out = 24, {}

    def rstr(p):
        n = struct.unpack_from("<Q", buf, p)[0]
        return buf[p + 8:p + 8 + n].decode("utf-8", "replace"), p + 8 + n

    def skip(t, p):
        if t in _SCALARS:
            return p + struct.calcsize(_SCALARS[t])
        if t == 8:
            n = struct.unpack_from("<Q", buf, p)[0]
            return p + 8 + n
        if t == 9:
            et, cnt = struct.unpack_from("<IQ", buf, p)
            p += 12
            if et in _SCALARS:
                return p + cnt * struct.calcsize(_SCALARS[et])
            for _ in range(cnt):
                p = skip(et, p)
            return p
        raise ValueError(f"bad GGUF value type {t}")

    for _ in range(n_kv):
        key, pos = rstr(pos)
        t = struct.unpack_from("<I", buf, pos)[0]
        pos += 4
        keep = key.startswith(want_prefixes) or key.endswith((".block_count", ".context_length",
               ".head_count_kv", ".head_count", ".embedding_length", ".key_length", ".full_attention_interval"))
        if keep and t in _SCALARS:
            out[key] = struct.unpack_from(_SCALARS[t], buf, pos)[0]
            pos += struct.calcsize(_SCALARS[t])
        elif keep and t == 8:
            out[key], pos = rstr(pos)
        else:
            pos = skip(t, pos)
    return out

def model_dims(meta, fallback):
    """(n_layer, n_kv_head, head_dim, n_ctx_train) from GGUF metadata, falling back to the profile."""
    arch = meta.get("general.architecture", "")
    g = lambda k: meta.get(f"{arch}.{k}")
    try:
        n_layer = int(g("block_count") or fallback["N_LAYER"])
        kvh = g("attention.head_count_kv")
        kvh = int(kvh) if isinstance(kvh, (int, float)) and kvh else fallback["N_KV_HEAD"]
        hd = g("attention.key_length")
        if not hd:
            emb, heads = g("embedding_length"), g("attention.head_count")
            hd = (int(emb) // int(heads)) if emb and heads else fallback["HEAD_DIM"]
        return n_layer, kvh, int(hd), int(g("context_length") or 0)
    except Exception:
        return fallback["N_LAYER"], fallback["N_KV_HEAD"], fallback["HEAD_DIM"], 0

def attn_layers(meta, n_layer, fallback_interval):
    """How many layers keep a KV cache. Hybrid models store `full_attention_interval` (every Nth layer is attention)."""
    arch = meta.get("general.architecture", "")
    itv = meta.get(f"{arch}.full_attention_interval") or fallback_interval
    try:
        itv = int(itv)
    except (TypeError, ValueError):
        itv = int(fallback_interval or 0)
    return max(1, n_layer // itv) if itv and itv > 1 else n_layer


# ---------------------------------------------------------------- GPU memory planner
KV_BYTES_PER_ELEM = {"f16": 2.0, "q8_0": 34 / 32, "q4_0": 18 / 32}   # ggml block sizes

def kv_mib(ctx, kv, n_layer, n_kv_head, head_dim):
    return 2 * n_layer * n_kv_head * head_dim * KV_BYTES_PER_ELEM[kv] * ctx / 1048576

def plan_candidates(model_mib, total_mib, ladder, target, kv_pref, dims, overhead_mib, min_free_mib, auto_order=None, extra_fn=None):
    """Ordered (ctx, kv) launch attempts. Largest context first; at equal context the best KV precision first.
    `extra_fn(ctx)` = additional MiB that grow with the window (the MTP draft's own cache).
    Entries predicted to fit come first; the rest (a safety net if the estimate is too pessimistic) follow."""
    kv_order = {"auto": auto_order or ["q8_0", "q4_0"], "f16": ["f16"], "q8_0": ["q8_0"], "q4_0": ["q4_0"]}[kv_pref]
    extra_fn = extra_fn or (lambda c: 0)
    sizes = [c for c in ladder if c <= target]
    fit, nofit = [], []
    for c in sizes:
        for kv in kv_order:
            need = model_mib + kv_mib(c, kv, *dims) + overhead_mib + min_free_mib + extra_fn(c)
            (fit if need <= total_mib else nofit).append((c, kv))
    return fit + [x for x in nofit if x[1] == kv_order[-1]][:3]

def spec_flags(mode, depth, draft_path):
    """llama-server arguments for speculative decoding, as (flag, value, ...) tuples. Values as on the model card."""
    if mode == "fast":
        return [("--spec-draft-model", draft_path), ("--spec-draft-ngl", "all"), ("--spec-type", "draft-mtp"),
                ("--spec-draft-n-max", depth), ("--spec-draft-p-min", 0)]
    if mode == "embedded":
        return [("--spec-type", "draft-mtp"), ("--spec-draft-n-max", depth)]
    return []

def degrade_spec(mode):
    return {"fast": "embedded", "embedded": "off"}.get(mode, "off")

def flag_supported(help_text, flag):
    """True if `flag` (e.g. '--spec-type') appears as an option in a `llama-server --help` text."""
    return re.search(r"(?<![\w-])" + re.escape(flag) + r"(?![\w-])", help_text or "") is not None

def tool_cap_chars(ctx):
    """Max characters one web-tool result may add to the conversation (~1/8 of the window, 3 chars/token)."""
    return int(min(30000, max(5000, ctx * 0.375)))


# ---------------------------------------------------------------- web UI tarball (any archive layout)
def extract_ui(tgz, dest):
    """Unpack llama-<tag>-ui.tar.gz into `dest`, whatever the top-level layout is: the directory that
    holds index.html (shallowest one) becomes `dest`."""
    import tarfile, shutil
    tmp = dest + ".tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp)
    with tarfile.open(tgz) as t:
        try:
            t.extractall(tmp, filter="data")
        except TypeError:                       # Python without extraction filters
            t.extractall(tmp)
    root, best = None, 10**6
    for r, _dirs, files in os.walk(tmp):
        if "index.html" in files:
            depth = os.path.relpath(r, tmp).count(os.sep)
            if depth < best:
                best, root = depth, r
    if root is None:
        shutil.rmtree(tmp, ignore_errors=True)
        raise RuntimeError("the UI archive has no index.html")
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(root, dest)
    shutil.rmtree(tmp, ignore_errors=True)
    return dest


# ---------------------------------------------------------------- tiny PNG for the vision self-test
def make_png_solid(w, h, rgb):
    """A valid w x h solid-colour PNG (no imaging library needed)."""
    import zlib
    def chunk(tag, data):
        c = struct.pack(">I", len(data)) + tag + data
        return c + struct.pack(">I", zlib.crc32(tag + data) & 0xFFFFFFFF)
    row = b"\x00" + bytes(rgb) * w
    raw = row * h
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))
# ==== PURE-END ====


LOG, TLOG = "/content/llama-server.log", "/content/tunnel.log"
CONFIG = PROFILE["CONFIG"]
WEB_SCRIPT, MCP_CONFIG = "/content/colab_web_mcp.py", "/content/colab_mcp.json"
TEMPLATE_FILE = "/content/chat_template_tools.jinja"
WEBUI_DIR = "/content/webui"
MiB = 1048576
LOCAL = requests.Session()
LOCAL.trust_env = False       # talk to the local server/tunnel directly, never through a proxy

def free_port(start):
    """First port from `start` that nothing else is using."""
    for p in range(start, start + 100):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sk:
            try:
                sk.bind(("127.0.0.1", p))
                return p
            except OSError:
                continue
    raise RuntimeError(f"No free port found near {start}")

def secret(name, typed):
    if typed.strip():
        return typed.strip()
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

def gpu_mem():
    """(used MiB, total MiB) summed over GPUs, or (None, None)."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
        used = total = 0
        for line in out.strip().splitlines():
            u, t = line.split(",")
            used += int(u); total += int(t)
        return (used, total) if total else (None, None)
    except Exception:
        return None, None

TOGGLE_JS = r'''/* Injected by the notebook (tpuchat-v2): "Allow tool calls" switch inside the chat box,
   right after the + button. Applies instantly - never reloads the page, so a running
   answer (text or reasoning) is never interrupted. The choice is saved per browser. */
(function () {
  "use strict";
  if (window.__tpuchatAllowTools) return;
  window.__tpuchatAllowTools = true;

  var STATE_KEY = "tpuchat.allowTools";
  var TOOL_NAMES = ["web_search", "web_fetch", "web_research"];
  /* localStorage lists the official UI reads exactly once, at startup */
  var STORE_KEYS = ["LlamaUi.alwaysAllowedTools", "LlamaCppWebui.alwaysAllowedTools"];
  /* permission keys the UI derives for these tools (toolKey(): server:/mcp-<id>:/custom:) */
  var PERM_KEYS = [];
  var JUNK_KEYS = [];
  TOOL_NAMES.forEach(function (n) {
    PERM_KEYS.push("server:" + n, "mcp-web:" + n, "mcp-web:" + n.replace(/^web_/, ""), "custom:" + n);
  });
  ["web_web_search", "web_web_fetch", "web_web_research"].forEach(function (n) {
    JUNK_KEYS.push("server:" + n, "mcp-web:" + n, "custom:" + n);
  });
  TOOL_NAMES.forEach(function (n) {
    JUNK_KEYS.push("browser:" + n, "mcp:" + n);
  });

  function isOn() {
    try { var v = localStorage.getItem(STATE_KEY); return v === null ? true : v === "1"; }
    catch (e) { return true; }
  }
  function readList(k) {
    try {
      var a = JSON.parse(localStorage.getItem(k) || "[]");
      return Array.isArray(a) ? a.filter(function (x) { return typeof x === "string"; }) : [];
    } catch (e) { return []; }
  }
  function writeList(k, list) {
    try { localStorage.setItem(k, JSON.stringify(list)); } catch (e) {}
  }
  function cleanAndWrite(k, set) {
    JUNK_KEYS.forEach(function (p) { delete set[p]; });
    writeList(k, Object.keys(set));
  }
  function syncPerms(on) {
    STORE_KEYS.forEach(function (k) {
      var set = {};
      readList(k).forEach(function (x) { set[x] = 1; });
      PERM_KEYS.forEach(function (p) { if (on) set[p] = 1; else delete set[p]; });
      cleanAndWrite(k, set);
    });
  }
  function allowSaved(tool) {
    var keys = ["server:" + tool, "mcp-web:" + tool, "mcp-web:" + tool.replace(/^web_/, ""), "custom:" + tool];
    STORE_KEYS.forEach(function (k) {
      var set = {};
      readList(k).forEach(function (x) { set[x] = 1; });
      keys.forEach(function (p) { set[p] = 1; });
      cleanAndWrite(k, set);
    });
  }
  /* runs before the UI boots: the app picks this list up when it initializes */
  syncPerms(isOn());

  /* -------------- live plumbing (no reload: streaming is never interrupted) -------------- */

  var sessionAll = false;            /* "allow this session" covers all web tools until OFF again */
  var seenCard = { tool: "", t: 0 }; /* built-in permission card the user may act on */

  function isWebTool(t) { return typeof t === "string" && /^web_/.test(t); }

  /* Fetch gate - active only while the switch is OFF. Covers the one case localStorage
     cannot: web tools the UI still holds in its in-memory allowed set from an earlier
     boot, which would otherwise run silently instead of asking. */
  var origFetch = window.fetch.bind(window);
  window.fetch = function () {
    try {
      var input = arguments[0], init = arguments[1];
      var url = typeof input === "string" ? input : (input && input.url) || "";
      var method = ((init && init.method) || (input && input.method) || "GET").toUpperCase();
      if (method === "POST" && /\/tools\/?$/.test(String(url).split("?")[0]) &&
          init && typeof init.body === "string") {
        var tool = null;
        try { tool = (JSON.parse(init.body) || {}).tool; } catch (e) {}
        if (isWebTool(tool) && !isOn() && !sessionAll &&
            !(seenCard.tool === tool && Date.now() - seenCard.t < 8000)) {
          return gate(tool, arguments);
        }
      }
    } catch (e) { /* never break the app */ }
    return origFetch.apply(null, arguments);
  };

  var activeFinish = null;   /* decision callback of the currently open prompt */
  var gateChain = Promise.resolve();
  function gate(tool, args) {
    var run = function () {
      return new Promise(function (resolve) {
        var settled = false;
        var signal = args[1] && args[1].signal;
        function finish(allow, remember) {
          if (settled) return;
          settled = true;
          activeFinish = null;
          hideGateCard();
          if (remember) sessionAll = true;
          if (remember === "saved") allowSaved(tool);
          if (allow) resolve(origFetch.apply(null, args));
          else resolve(new Response(JSON.stringify({
                   error: "Blocked by the user: the Allow tool calls switch is OFF in the chat box. Turn it ON (green) to run web tools without asking."
                 }), { status: 403, statusText: "Blocked",
                      headers: { "Content-Type": "application/json" } }));
        }
        function abortNow() {
          if (settled) return;
          settled = true;
          activeFinish = null;
          hideGateCard();
          resolve(Promise.reject(new DOMException("The tool call was aborted.", "AbortError")));
        }
        if (signal) {
          if (signal.aborted) { abortNow(); return; }
          signal.addEventListener("abort", abortNow, { once: true });
        }
        showGateCard(tool, finish);
      });
    };
    gateChain = gateChain.then(run, run);
    return gateChain;
  }

  function esc(s) {
    return String(s).replace(/[&<>"']/g, function (c) {
      return { "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;" }[c];
    });
  }

  var gateEl = null;
  function showGateCard(tool, finish) {
    hideGateCard();
    var d = document.createElement("div");
    d.id = "tpu-gate-card";
    d.setAttribute("role", "dialog");
    d.setAttribute("aria-label", "Tool call permission");
    d.innerHTML = '<div class="t">The model wants to run <b>' + esc(tool) + "</b></div>" +
      '<div class="s">The "Allow tool calls" switch is OFF, so this call needs your OK.</div>' +
      '<div class="row">' +
      '<button type="button" data-a="once">Allow once</button>' +
      '<button type="button" data-a="session">Allow this session</button>' +
      '<button type="button" data-a="saved">Always allow</button>' +
      '<button type="button" data-a="deny" class="deny">Deny</button>' +
      "</div>";
    d.addEventListener("click", function (e) {
      var b = e.target && e.target.closest ? e.target.closest("button[data-a]") : null;
      if (!b) return;
      var a = b.getAttribute("data-a");
      finish(a !== "deny", a === "session" ? "session" : (a === "saved" ? "saved" : null));
    });
    (document.body || document.documentElement).appendChild(d);
    gateEl = d;
    activeFinish = finish;
  }
  function hideGateCard() {
    if (gateEl) { try { gateEl.remove(); } catch (e) {} gateEl = null; }
  }
  document.addEventListener("keydown", function (e) {
    if (e.key === "Escape" && activeFinish) activeFinish(false, null);
  }, true);

  /* Watch the built-in permission card ("Allow use of <tool> ..."). While the switch is
     ON, approve it automatically (only needed when the in-memory set is missing keys,
     e.g. the switch was turned ON mid-session). While OFF, just remember it so a call
     the user just approved by hand is never asked twice. */
  function scanForCard() {
    var btns = document.getElementsByTagName("button");
    for (var i = 0; i < btns.length; i++) {
      var b = btns[i];
      if ((b.textContent || "").trim() !== "Allow once") continue;
      var root = b.parentElement, cardText = "";
      for (var k = 0; k < 7 && root; k++, root = root.parentElement) {
        cardText = root.textContent || "";
        if (cardText.indexOf("Allow use of ") >= 0) break;
      }
      if (cardText.indexOf("Allow use of ") < 0) continue;
      var m = cardText.match(/Allow use of\s+([A-Za-z0-9_.-]+)/);
      if (m) { seenCard.tool = m[1]; seenCard.t = Date.now(); }
      if (isOn() && !b.dataset.tpuDone) {
        b.dataset.tpuDone = "1";
        try { b.click(); } catch (e) {}
      }
    }
  }

  /* -------------- the switch, rendered inside the composer after the + button -------------- */

  var CSS_TEXT =
    "#tpu-allow-tools{display:inline-flex;align-items:center;gap:7px;height:32px;padding:0 12px;" +
    "border-radius:999px;border:1px solid rgba(148,163,184,.35);background:rgba(100,116,139,.14);" +
    "color:inherit;opacity:.85;font:600 12px/1 ui-sans-serif,system-ui,-apple-system,sans-serif;" +
    "cursor:pointer;user-select:none;white-space:nowrap;flex:0 0 auto;" +
    "transition:border-color .15s,background .15s,opacity .15s}" +
    "#tpu-allow-tools:hover{opacity:1;border-color:rgba(148,163,184,.6)}" +
    "#tpu-allow-tools .dot{width:8px;height:8px;border-radius:50%;background:#64748b;flex:0 0 auto;" +
    "transition:background .15s,box-shadow .15s}" +
    "#tpu-allow-tools.on{border-color:rgba(34,197,94,.55);background:rgba(34,197,94,.13);opacity:1}" +
    "#tpu-allow-tools.on .dot{background:#22c55e;box-shadow:0 0 7px rgba(34,197,94,.8)}" +
    "#tpu-allow-tools.on .lbl{color:#16a34a}" +
    "#tpu-allow-tools .lbl-short{display:none}" +
    "#tpu-gate-card{position:fixed;left:50%;bottom:96px;transform:translateX(-50%);z-index:2147483000;" +
    "width:min(440px,calc(100vw - 24px));padding:14px 16px;border-radius:14px;color:#e2e8f0;" +
    "background:#0f172a;border:1px solid rgba(148,163,184,.3);box-shadow:0 12px 40px rgba(0,0,0,.45);" +
    "font:400 13px/1.5 ui-sans-serif,system-ui,-apple-system,sans-serif;animation:tpu-pop .16s ease-out}" +
    "#tpu-gate-card .t{font-weight:600;margin-bottom:2px}" +
    "#tpu-gate-card .s{opacity:.75;margin-bottom:10px}" +
    "#tpu-gate-card .row{display:flex;flex-wrap:wrap;gap:8px}" +
    "#tpu-gate-card button{all:unset;cursor:pointer;padding:6px 12px;border-radius:8px;" +
    "font:600 12px/1 ui-sans-serif,system-ui,sans-serif;color:inherit;" +
    "border:1px solid rgba(148,163,184,.4);background:rgba(148,163,184,.12)}" +
    "#tpu-gate-card button:hover{background:rgba(148,163,184,.24)}" +
    "#tpu-gate-card button.deny{border-color:rgba(239,68,68,.5);color:#f87171}" +
    "@keyframes tpu-pop{from{opacity:0;transform:translateX(-50%) translateY(6px)}" +
    "to{opacity:1;transform:translateX(-50%) translateY(0)}}" +
    "@media (prefers-color-scheme: dark){#tpu-allow-tools.on .lbl{color:#4ade80}}" +
    "@media (prefers-color-scheme: light){#tpu-gate-card{background:#fff;color:#0f172a;" +
    "border-color:#cbd5e1;box-shadow:0 12px 40px rgba(15,23,42,.18)}}" +
    "@media (max-width:640px){#tpu-allow-tools{height:28px;padding:0 9px;gap:5px;font-size:11px}" +
    "#tpu-allow-tools .lbl-long{display:none}#tpu-allow-tools .lbl-short{display:inline}" +
    "#tpu-gate-card{bottom:84px}}";

  var chip = null;
  function paint() {
    if (!chip) return;
    var on = isOn();
    chip.classList.toggle("on", on);
    chip.setAttribute("aria-checked", on ? "true" : "false");
    var long = chip.querySelector(".lbl-long"), short = chip.querySelector(".lbl-short");
    if (long) long.textContent = "Allow tool calls: " + (on ? "ON" : "OFF");
    if (short) short.textContent = "Tools: " + (on ? "ON" : "OFF");
  }
  function buildChip() {
    try {
      var css = document.createElement("style");
      css.id = "tpu-allow-tools-css";
      css.textContent = CSS_TEXT;
      document.head.appendChild(css);
    } catch (e) {}
    var b = document.createElement("button");
    b.id = "tpu-allow-tools";
    b.type = "button";
    b.setAttribute("role", "switch");
    b.setAttribute("aria-label", "Allow tool calls");
    b.title = "ON: web tools (search / fetch) run without asking. OFF: each web tool call asks first. " +
      "Applies instantly - no reload, typing and generating continue undisturbed.";
    b.innerHTML = '<span class="dot"></span><span class="lbl lbl-long"></span><span class="lbl lbl-short"></span>';
    /* keep the composer textarea focused and never trigger the + menu or a form submit */
    ["mousedown", "pointerdown", "touchstart", "touchend"].forEach(function (ev) {
      b.addEventListener(ev, function (e) { e.preventDefault(); e.stopPropagation(); });
    });
    b.addEventListener("click", function (e) {
      e.preventDefault();
      e.stopPropagation();
      var next = !isOn();
      try { localStorage.setItem(STATE_KEY, next ? "1" : "0"); } catch (err) {}
      syncPerms(next);
      if (!next) sessionAll = false;   /* a fresh OFF period asks again */
      paint();
      /* turning ON while a call is waiting for permission: let it through */
      if (next && activeFinish) activeFinish(true, "session");
      try {
        window.dispatchEvent(new StorageEvent("storage", { key: STATE_KEY, newValue: next ? "1" : "0" }));
      } catch (err) {}
    });
    paint();
    return b;
  }
  function ensureChip() {
    var host = document.querySelector("button.file-upload-button");
    if (!host || !host.isConnected) return false;
    if (!chip) chip = buildChip();
    /* preferred spot: the left group that holds the + button, next to it */
    var group = host.closest(".mr-auto") || host.parentElement;
    if (!group) return false;
    if (chip.isConnected && chip.parentElement === group) return true;
    try { group.appendChild(chip); paint(); } catch (e) { return false; }
    return true;
  }

  window.addEventListener("storage", function (e) {
    if (e && e.key === STATE_KEY) { try { paint(); } catch (err) {} }
  });

  /* -------------- keep-alive: survive Svelte re-renders and route changes -------------- */

  var scanTimer = 0, chipTimer = 0;
  function scheduleScan() {
    if (scanTimer) return;
    scanTimer = setTimeout(function () { scanTimer = 0; try { scanForCard(); } catch (e) {} }, 120);
  }
  function scheduleChip() {
    if (chipTimer) return;
    chipTimer = setTimeout(function () { chipTimer = 0; try { ensureChip(); } catch (e) {} }, 150);
  }
  function start() {
    try {
      var mo = new MutationObserver(function (muts) {
        scheduleChip();
        for (var i = 0; i < muts.length; i++) {
          var nodes = muts[i].addedNodes;
          for (var j = 0; j < nodes.length; j++) {
            var n = nodes[j];
            if (n.nodeType === 1 && (n.textContent || "").indexOf("Allow use of ") >= 0) {
              scheduleScan();
              return;
            }
          }
        }
      });
      mo.observe(document.documentElement, { childList: true, subtree: true });
    } catch (e) {}
    setInterval(function () {
      try { ensureChip(); } catch (e) {}
      try { scanForCard(); } catch (e) {}
    }, 1500);
    try { ensureChip(); } catch (e) {}
    console.info("[tpuchat] Allow tool calls switch installed (in the chat box, next to +)");
  }
  if (document.readyState === "loading") document.addEventListener("DOMContentLoaded", start);
  else start();
})();'''


WEB_MCP_SOURCE = r'''#!/usr/bin/env python3
"""Colab web tools: a tiny stdio MCP server for llama-server (--mcp-servers-config).

Gives the model web_search / web_fetch (and web_research when a Parallel key is set), backed by:
  parallel   - Parallel MCP (https://mcp.parallel.ai/mcp), free without a key
  exa        - Exa MCP (https://mcp.exa.ai/mcp), free tier is rate limited, key optional
  duckduckgo - DuckDuckGo search (ddgs) + direct page download, no key
If the chosen backend fails (rate limit, network, timeout) the next one is tried, and EVERY call is
bounded by a hard deadline (WEB_DEADLINE, default 80 s): a slow backend can never leave the chat
hanging, and the answer always comes back before a proxy (Cloudflare: 100 s) would cut the request.

Settings come from environment variables set by the notebook:
  WEB_BACKEND        parallel | exa | duckduckgo        (default parallel)
  PARALLEL_API_KEY   optional, higher limits + adds web_research
  EXA_API_KEY        optional
  WEB_FALLBACK       1 = try other backends on failure  (default 1)
  WEB_MAX_CHARS      cap on characters returned per call (set by the notebook from the context size)
  WEB_DEADLINE       hard limit in seconds for one tool call (default 80)
Only JSON-RPC goes to stdout; diagnostics go to stderr.
"""
import json, os, re, sys, time, html, threading, urllib.request, urllib.error

BACKEND = os.environ.get("WEB_BACKEND", "parallel").strip().lower() or "parallel"
PARALLEL_KEY = os.environ.get("PARALLEL_API_KEY", "").strip()
EXA_KEY = os.environ.get("EXA_API_KEY", "").strip()
FALLBACK = os.environ.get("WEB_FALLBACK", "1") != "0"
MAX_CHARS = int(os.environ.get("WEB_MAX_CHARS", "12000") or 12000)
DEADLINE = float(os.environ.get("WEB_DEADLINE", "80") or 80)
HTTP_TIMEOUT = 25
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130 Safari/537.36"


def log(*a):
    print("[colab-web]", *a, file=sys.stderr, flush=True)


# ----------------------------------------------------------------- remote MCP client
class RemoteMCP:
    """Minimal Streamable-HTTP MCP client (JSON or SSE replies)."""

    def __init__(self, url, headers=None, timeout=HTTP_TIMEOUT):
        self.url, self.timeout, self.sid = url, timeout, None
        self.headers = {"Content-Type": "application/json",
                        "Accept": "application/json, text/event-stream",
                        "User-Agent": "colab-llama/5"}
        self.headers.update(headers or {})
        self._id = 0

    def _post(self, payload, want_reply=True):
        h = dict(self.headers)
        if self.sid:
            h["Mcp-Session-Id"] = self.sid
        req = urllib.request.Request(self.url, data=json.dumps(payload).encode("utf-8"),
                                     headers=h, method="POST")
        try:
            with urllib.request.urlopen(req, timeout=self.timeout) as r:
                sid = r.headers.get("Mcp-Session-Id")
                if sid:
                    self.sid = sid
                ctype = (r.headers.get("Content-Type") or "").lower()
                body = r.read().decode("utf-8", errors="replace")   # always UTF-8 (no mojibake)
        except urllib.error.HTTPError as e:
            detail = ""
            try:
                detail = e.read().decode("utf-8", errors="replace")[:500]
            except Exception:
                pass
            raise RuntimeError(f"HTTP {e.code} from {self.url}: {detail}")
        if not want_reply:
            return None
        msgs = []
        if "event-stream" in ctype:
            data_lines = []
            for line in body.splitlines() + [""]:          # SSE: blank line ends an event
                if line.startswith("data:"):
                    data_lines.append(line[5:].lstrip(" "))
                elif line.strip() == "":
                    if data_lines:
                        try:
                            msgs.append(json.loads("\n".join(data_lines)))
                        except ValueError:
                            pass
                        data_lines = []
        elif body.strip():
            j = json.loads(body)
            msgs = j if isinstance(j, list) else [j]
        for m in msgs:
            if isinstance(m, dict) and m.get("id") == payload.get("id"):
                return m
        for m in msgs:                                      # some servers reply with id null on errors
            if isinstance(m, dict) and "error" in m:
                return m
        raise RuntimeError("No reply from MCP server")

    def _rpc(self, method, params=None):
        self._id += 1
        reply = self._post({"jsonrpc": "2.0", "id": self._id, "method": method, "params": params or {}})
        if "error" in reply:
            err = reply["error"]
            raise RuntimeError(err.get("message", str(err)) if isinstance(err, dict) else str(err))
        return reply.get("result", {})

    def call(self, tool, args):
        self._rpc("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                                 "clientInfo": {"name": "colab-llama", "version": "5"}})
        try:
            self._post({"jsonrpc": "2.0", "method": "notifications/initialized"}, want_reply=False)
        except Exception:
            pass
        res = self._rpc("tools/call", {"name": tool, "arguments": args})
        text = result_text(res)
        if res.get("isError"):
            raise RuntimeError(text or "tool error")
        return text


def result_text(res):
    parts = []
    for c in res.get("content") or []:
        if isinstance(c, dict) and c.get("type") == "text" and c.get("text"):
            parts.append(c["text"])
    if not parts and res.get("structuredContent") is not None:
        parts.append(json.dumps(res["structuredContent"], ensure_ascii=False, indent=1))
    return "\n".join(parts).strip()


# ----------------------------------------------------------------- backends  (each takes: args, timeout)
def _queries(a):
    q = a.get("query") or ""
    qs = a.get("search_queries") or []
    if isinstance(qs, str):
        qs = [qs]
    qs = [str(x).strip() for x in qs if str(x).strip()]
    if q.strip() and q.strip() not in qs:
        qs.insert(0, q.strip())
    return qs[:5]


def _urls(a):
    u = a.get("urls") or a.get("url") or []
    if isinstance(u, str):
        u = re.split(r"[\s,]+", u)
    out = []
    for x in u:
        x = str(x).strip().strip("<>\"'")
        if x and not re.match(r"^https?://", x, re.I):
            x = "https://" + x
        if x and x not in out:
            out.append(x)
    return out[:10]


def _phdr():
    return {"Authorization": f"Bearer {PARALLEL_KEY}"} if PARALLEL_KEY else {}


def parallel_search(a, timeout):
    qs = _queries(a)
    objective = (a.get("objective") or "").strip() or (qs[0] if qs else "")
    return RemoteMCP("https://mcp.parallel.ai/mcp", _phdr(), timeout).call(
        "web_search", {"objective": objective[:4000], "search_queries": qs or [objective[:200]]})


def parallel_fetch(a, timeout):
    args = {"urls": _urls(a)}
    if a.get("objective"):
        args["objective"] = str(a["objective"])[:200]
    if a.get("full_content"):
        args["full_content"] = True
    return RemoteMCP("https://mcp.parallel.ai/mcp", _phdr(), timeout).call("web_fetch", args)


def parallel_research(a, timeout):
    args = {"input": str(a.get("question") or a.get("input") or a.get("query") or "")[:20000]}
    return RemoteMCP("https://mcp.parallel.ai/mcp", _phdr(), timeout).call("web_research", args)


def _exa(timeout):
    hdr = {"Authorization": f"Bearer {EXA_KEY}"} if EXA_KEY else {}
    return RemoteMCP("https://mcp.exa.ai/mcp", hdr, timeout)


def exa_search(a, timeout):
    qs = _queries(a)
    args = {"query": " ".join(qs[:1]) or a.get("objective", ""), "numResults": 8}
    if a.get("objective"):
        args["objective"] = str(a["objective"])[:4000]
    return _exa(timeout).call("web_search_exa", args)


def exa_fetch(a, timeout):
    return _exa(timeout).call("web_fetch_exa", {"urls": _urls(a),
                                                "maxCharacters": 60000 if a.get("full_content") else 8000})


def ddg_search(a, timeout):
    try:
        from ddgs import DDGS
    except ImportError:
        raise RuntimeError("ddgs is not installed (pip install ddgs)")
    out, seen = [], set()
    for q in _queries(a)[:3] or [a.get("objective", "")]:
        try:
            hits = DDGS(timeout=min(15, timeout)).text(q, max_results=8) or []
        except TypeError:                       # very old ddgs without the timeout kwarg
            hits = DDGS().text(q, max_results=8) or []
        for r in hits:
            url = r.get("href") or r.get("url")
            if not url or url in seen:
                continue
            seen.add(url)
            out.append(f"## {r.get('title','')}\nURL: {url}\n{r.get('body','')}")
    if not out:
        raise RuntimeError("DuckDuckGo returned no results")
    return "\n\n".join(out)


def _download(url, timeout):
    req = urllib.request.Request(url, headers={"User-Agent": UA, "Accept": "text/html,*/*;q=0.8"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        raw = r.read(3_000_000)
        cs = r.headers.get_content_charset() or "utf-8"
        ctype = (r.headers.get("Content-Type") or "").lower()
    try:
        txt = raw.decode(cs, errors="replace")
    except LookupError:
        txt = raw.decode("utf-8", errors="replace")
    if "html" not in ctype and "<html" not in txt[:2000].lower():
        return txt
    try:
        import trafilatura
        ext = trafilatura.extract(txt, url=url, include_links=False, include_tables=True,
                                  output_format="markdown")
        if ext and len(ext) > 200:
            return ext
    except Exception:
        pass
    txt = re.sub(r"(?is)<(script|style|noscript|svg|head)[^>]*>.*?</\1>", " ", txt)
    txt = re.sub(r"(?s)<[^>]+>", " ", txt)
    return re.sub(r"[ \t\r\f\v]+", " ", re.sub(r"\n\s*\n+", "\n\n", html.unescape(txt))).strip()


def direct_fetch(a, timeout):
    urls, parts = _urls(a), []
    if not urls:
        raise RuntimeError("no valid URL given")
    per = max(3000, MAX_CHARS // len(urls))
    for u in urls[:5]:
        try:
            parts.append(f"# {u}\n{_download(u, min(timeout, 20))[:per]}")
        except Exception as e:
            parts.append(f"# {u}\n(could not load: {e})")
    return "\n\n".join(parts)


SEARCH = {"parallel": parallel_search, "exa": exa_search, "duckduckgo": ddg_search}
FETCH = {"parallel": parallel_fetch, "exa": exa_fetch, "duckduckgo": direct_fetch}


def _call_bounded(fn, args, seconds):
    """Run fn(args, seconds) in a worker thread and give up after `seconds` (the worker is a daemon)."""
    box = {}

    def work():
        try:
            box["r"] = fn(args, seconds)
        except BaseException as e:          # noqa: BLE001 - reported to the caller
            box["e"] = e

    t = threading.Thread(target=work, daemon=True)
    t.start()
    t.join(max(0.5, seconds))
    if t.is_alive():
        raise TimeoutError(f"no answer within {seconds:.0f}s")
    if "e" in box:
        raise box["e"]
    return box["r"]


def run_with_fallback(table, args, budget=None):
    """Try the chosen backend, then the others; the WHOLE call is bounded by `budget` seconds."""
    t0 = time.time()
    budget = DEADLINE if budget is None else budget
    order = [BACKEND] + ([b for b in ("parallel", "duckduckgo", "exa") if b != BACKEND] if FALLBACK else [])
    errors = []
    todo = [b for b in order if b in table]
    for i, b in enumerate(todo):
        left = budget - (time.time() - t0)
        if left < 3:
            errors.append(f"{b}: skipped (out of time)")
            break
        # fair slice: a hung backend must never starve the fallbacks that come after it
        per = min(HTTP_TIMEOUT, max(3.0, (left - 1) / (len(todo) - i)))
        try:
            text = _call_bounded(table[b], args, per)
            if text and text.strip():
                note = "" if b == BACKEND else f"[results from {b}; {BACKEND} was unavailable]\n"
                return note + text
            raise RuntimeError("empty result")
        except Exception as e:              # noqa: BLE001
            msg = str(e) or type(e).__name__
            log(f"{b} failed: {msg[:300]}")
            errors.append(f"{b}: {msg[:160]}")
    raise RuntimeError("all web backends failed - " + " | ".join(errors))


def trim(text, limit):
    if len(text) <= limit:
        return text
    cut = text[:limit]
    nl = cut.rfind("\n")
    if nl > limit * 0.7:
        cut = cut[:nl]
    return cut + "\n\n[...truncated to fit the context window]"


# ----------------------------------------------------------------- MCP server side
TOOLS = [
    {"name": "search",
     "description": ("Search the web for current information. Returns titles, URLs and text excerpts. "
                     "Use for anything recent, factual, or that you are unsure about."),
     "inputSchema": {"type": "object", "properties": {
         "query": {"type": "string", "description": "Short keyword search query, 3-8 words."},
         "objective": {"type": "string", "description": "Optional: one sentence on what you want to find."}},
         "required": ["query"]}},
    {"name": "fetch",
     "description": "Read one or more web pages (e.g. URLs from web_search) and return their main text.",
     "inputSchema": {"type": "object", "properties": {
         "urls": {"type": "array", "items": {"type": "string"}, "description": "Page URLs to read (1-3)."},
         "objective": {"type": "string", "description": "Optional: what to look for on the pages."},
         "full_content": {"type": "boolean", "description": "Return the whole page instead of the relevant parts."}},
         "required": ["urls"]}},
]
if BACKEND == "parallel" and PARALLEL_KEY:
    TOOLS.append({"name": "research",
                  "description": ("Deep research on ONE broad question; returns a cited answer. Slow (up to ~80 s) "
                                  "and uses credits - prefer search + fetch unless the question needs many sources."),
                  "inputSchema": {"type": "object", "properties": {
                      "question": {"type": "string", "description": "The research question."}},
                      "required": ["question"]}})


def handle(msg):
    method, mid = msg.get("method"), msg.get("id")
    if mid is None:
        return None                                          # notification
    if method == "initialize":
        pv = (msg.get("params") or {}).get("protocolVersion") or "2025-06-18"
        return {"protocolVersion": pv, "capabilities": {"tools": {"listChanged": False}},
                "serverInfo": {"name": "colab-web", "version": "5.0"}}
    if method == "ping":
        return {}
    if method == "tools/list":
        return {"tools": TOOLS}
    if method == "tools/call":
        p = msg.get("params") or {}
        name, args = p.get("name"), p.get("arguments") or {}
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except ValueError:
                args = {"query": args}
        if not isinstance(args, dict):
            args = {"query": str(args)}
        try:
            if name in ("search", "web_search"):
                if not _queries(args) and not args.get("objective"):
                    raise RuntimeError("web_search needs a 'query'")
                text = run_with_fallback(SEARCH, args)
            elif name in ("fetch", "web_fetch"):
                if not _urls(args):
                    raise RuntimeError("web_fetch needs at least one URL in 'urls'")
                text = run_with_fallback(FETCH, args)
            elif name in ("research", "web_research"):
                if not PARALLEL_KEY:
                    raise RuntimeError("web_research needs a Parallel API key")
                text = _call_bounded(parallel_research, args, max(10, DEADLINE - 3))
            else:
                raise RuntimeError(f"unknown tool {name}")
            return {"content": [{"type": "text", "text": trim(text, MAX_CHARS)}], "isError": False}
        except Exception as e:                  # noqa: BLE001
            note = ("Web tool error: " + (str(e) or type(e).__name__) +
                    "\n(Do not repeat the same call. Either try ONE differently-worded query, or answer now "
                    "from your own knowledge and say that the live lookup failed.)")
            return {"content": [{"type": "text", "text": note}], "isError": True}
    raise KeyError(method)


def main():
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            msg = json.loads(line)
        except ValueError:
            continue
        msgs = msg if isinstance(msg, list) else [msg]
        for m in msgs:
            if not isinstance(m, dict):
                continue
            try:
                res = handle(m)
                if res is None:
                    continue
                out = {"jsonrpc": "2.0", "id": m.get("id"), "result": res}
            except KeyError:
                out = {"jsonrpc": "2.0", "id": m.get("id"),
                       "error": {"code": -32601, "message": f"Method not found: {m.get('method')}"}}
            except Exception as e:              # noqa: BLE001
                out = {"jsonrpc": "2.0", "id": m.get("id"), "error": {"code": -32603, "message": str(e)}}
            sys.stdout.write(json.dumps(out, ensure_ascii=False) + "\n")
            sys.stdout.flush()


if __name__ == "__main__":
    main()
'''


def patch_webui(index_path):
    try:
        with open(index_path, encoding="utf-8") as f:
            html = f.read()
        tag = "<script>" + TOGGLE_JS + "</script>\n"
        if "</head>" in html:
            html = html.replace("</head>", tag + "</head>", 1)
        elif "<body" in html:
            html = html.replace("<body", tag + "<body", 1)
        else:
            html = tag + html
        with open(index_path, "w", encoding="utf-8") as f:
            f.write(html)
        return True
    except OSError:
        return False

def ensure_webui(cfg):
    """Copy the official chat UI (prepared by Cell 1, or downloaded now) and add the Allow-tool-calls switch."""
    src = cfg.get("webui_src") or "/content/webui_src"
    try:
        if not os.path.isfile(os.path.join(src, "index.html")):
            tag = cfg.get("ui_tag") or cfg.get("tag")
            print(f"\u23f3 Downloading the chat UI ({tag})...")
            r = requests.get(f"https://github.com/ggml-org/llama.cpp/releases/download/{tag}/llama-{tag}-ui.tar.gz", timeout=180)
            r.raise_for_status()
            with open("/content/webui.tar.gz", "wb") as f:
                f.write(r.content)
            extract_ui("/content/webui.tar.gz", src)
        shutil.rmtree(WEBUI_DIR, ignore_errors=True)
        shutil.copytree(src, WEBUI_DIR)
        if patch_webui(os.path.join(WEBUI_DIR, "index.html")):
            print("\u2705 Chat UI ready (with the Allow tool calls switch)")
        else:
            print("\u26a0\ufe0f Chat UI ready, but the Allow tool calls switch could not be added (the UI asks per tool call instead).")
        return WEBUI_DIR
    except Exception as e:                      # noqa: BLE001
        raise RuntimeError(f"Could not prepare the chat UI ({e}). Check your internet connection and run Cell 2 again.") from e

# ---------------- system prompt (identity + web workflow; thinking is the model's own, no prompt protocol) ----------------
def build_system_message():
    today = datetime.date.today().strftime("%A, %B %d, %Y")
    vision = " You can also read images that the user attaches." if VISION_STATE != "off" else ""
    parts = [PROFILE["IDENTITY"].format(today=today, vision=vision)]
    if WEB_BACKEND != "off":
        guide = ("WEB TOOLS: you can call web_search (short keyword query, 3-8 words) and web_fetch (read specific URLs). "
                 "Use them for anything recent, factual or uncertain. Workflow: search, then fetch the 1-3 most relevant pages "
                 "if the excerpts are not enough, then WRITE THE ANSWER. Make at most about 6 tool calls for one question, never "
                 "repeat an identical call, and ALWAYS finish with a complete written answer that cites the URLs you used - "
                 "never stop right after a tool result. If a tool returns an error or times out, say so briefly and continue "
                 "with what you know, or try one differently-worded query.")
        if PKEY and WEB_BACKEND == "parallel":
            guide += " web_research is slow and uses credits: only for broad questions that need many sources."
        parts.append(guide)
    return "\n\n".join(parts)

# ---------------- parse the form ----------------
_m = re.match(r"\d+", CONTEXT_WINDOW.strip())
TARGET_CONTEXT = int(_m.group()) if _m else PROFILE["CTX_LADDER"][0]      # "Max ..." -> the top of the ladder
KV_PREF = KV_CACHE.split()[0].lower()
if KV_PREF not in ("auto", "f16", "q8_0", "q4_0"):
    KV_PREF = "auto"
REASONING_KEY = REASONING.split()[0].lower()
if REASONING_KEY not in ("max", "high", "medium", "low", "off"):
    REASONING_KEY = "max"
REASONING_BUDGET = {"max": -1, "high": 16384, "medium": 4096, "low": 1024, "off": 0}[REASONING_KEY]
REASONING_EFFORT = {"max": "xhigh", "high": "xhigh", "medium": "medium", "low": "low", "off": ""}[REASONING_KEY]
WEB_BACKEND = {"parallel": "parallel", "exa": "exa", "duckduckgo": "duckduckgo"}.get(WEB_SEARCH.split()[0].lower(), "off")
LINK_MODE = PUBLIC_LINK.split()[0].lower()
if LINK_MODE not in ("cloudflare", "off"):
    LINK_MODE = "cloudflare"
VISION_PREF = "off" if VISION_MODE.startswith("Off") else ("cpu" if "CPU" in VISION_MODE else "gpu")
SPEC_PREF = {"off": "off", "embedded": "embedded", "fastmtp": "fast"}.get(SPECULATIVE.split()[0].lower(), "off")

# ---------------- Qwen's recommended sampling differs between thinking and non-thinking ----------------
if REASONING_KEY == "off":
    TEMP, TOP_P, TOP_K, MIN_P = 0.7, 0.8, 20, 0.0    # Qwen3.8 NON-thinking mode
    _presence = 1.5
else:
    TEMP, TOP_P, TOP_K, MIN_P = 1.0, 0.95, 20, 0.0   # Qwen3.8 THINKING mode (the default)
    _presence = 0.0
REPEAT_PENALTY = 1.0                                 # the model card uses no repetition penalty
if PRESENCE_PENALTY is None:
    PRESENCE_PENALTY = _presence

LOG_NOTES = []                                       # things Cell 2 changed on its own; printed at start-up
if not os.path.exists(CONFIG):
    raise RuntimeError("Run Cell 1 first.")
cfg = json.load(open(CONFIG))
BIN, MODEL_PATH, CF, QUANT = cfg["bin"], cfg["model"], cfg["cloudflared"], cfg["quant"]
for _p in (BIN, MODEL_PATH):
    if not os.path.exists(_p):
        raise RuntimeError(f"Missing {_p}. Run Cell 1 again.")
if LINK_MODE == "cloudflare" and not os.path.exists(CF):
    raise RuntimeError("cloudflared is missing. Run Cell 1 again (or set PUBLIC_LINK to Off).")

PKEY = secret("PARALLEL_API_KEY", PARALLEL_API_KEY) if WEB_BACKEND != "off" else ""
EKEY = secret("EXA_API_KEY", EXA_API_KEY) if WEB_BACKEND != "off" else ""

# ---------------- model facts (from the GGUF itself) ----------------
try:
    META = read_gguf_meta(MODEL_PATH)
except Exception as _e:                         # noqa: BLE001
    print(f"\u26a0\ufe0f Could not read the GGUF header ({_e}); using built-in model facts.")
    META = {}
N_LAYER, N_KV_HEAD, HEAD_DIM, CTX_TRAIN = model_dims(META, PROFILE)
N_ATTN_LAYER = attn_layers(META, N_LAYER, PROFILE.get("FULL_ATTN_INTERVAL"))   # hybrid model: only these layers keep a KV cache
DIMS = (N_ATTN_LAYER, N_KV_HEAD, HEAD_DIM)
DRAFT_KV_BYTES = 2 * N_KV_HEAD * HEAD_DIM * 2   # the MTP head is one attention layer; its cache per token, 16-bit (worst case)
MODEL_MIB = os.path.getsize(MODEL_PATH) / MiB
ALIAS = f"{PROFILE['NAME']}-{QUANT}"

# ---------------- what this llama-server build understands ----------------
HELP_TEXT = ""
def load_help():
    global HELP_TEXT
    try:
        p = subprocess.run([BIN, "--help"], capture_output=True, text=True, timeout=90)
        HELP_TEXT = (p.stdout or "") + "\n" + (p.stderr or "")
    except Exception:                           # noqa: BLE001
        HELP_TEXT = ""
load_help()

def has_flag(flag):
    """True unless this build's --help clearly lacks the flag (an unreadable help text never blocks anything)."""
    return len(HELP_TEXT) < 2000 or flag_supported(HELP_TEXT, flag)

# ---------------- vision (image input through the mmproj file) ----------------
MMPROJ = cfg.get("mmproj", "")
VISION_STATE = "off"                            # "gpu" -> "cpu" -> "off": degraded automatically if it fails
if VISION_PREF != "off" and MMPROJ and os.path.exists(MMPROJ):
    VISION_STATE = VISION_PREF
elif VISION_PREF != "off":
    LOG_NOTES.append("Vision was requested but the image encoder file is missing - re-run Cell 1 with DOWNLOAD_VISION ticked. Continuing text-only.")
VISION = VISION_STATE != "off"

# ---------------- speculative decoding (MTP) ----------------
DRAFT = cfg.get("draft", "")
PATCHED = bool(cfg.get("fastmtp_patched"))
SPEC_STATE = SPEC_PREF                          # "fast" -> "embedded" -> "off": degraded automatically if it fails
def spec_problem(mode):
    """'' if this build can run `mode`, else why not."""
    if mode == "off":
        return ""
    need = ["--spec-type", "--spec-draft-n-max"] + (["--spec-draft-model"] if mode == "fast" else [])
    if mode == "fast" and not (DRAFT and os.path.exists(DRAFT)):
        return "the FastMTP draft file is missing (use the FastMTP option in Cell 1)"
    if mode == "fast" and not PATCHED:
        return "this server was not built with the FastMTP patch (use the FastMTP option in Cell 1)"
    if not all(has_flag(f) for f in need) or (len(HELP_TEXT) >= 2000 and "mtp" not in HELP_TEXT.lower()):
        return "this llama.cpp build does not offer MTP speculative decoding"
    return ""
while SPEC_STATE != "off" and spec_problem(SPEC_STATE):
    _nxt = degrade_spec(SPEC_STATE)
    LOG_NOTES.append(f"{'FastMTP' if SPEC_STATE == 'fast' else 'MTP'} is not available: {spec_problem(SPEC_STATE)}. "
                     + ("Using the built-in MTP instead." if _nxt == "embedded" else "Running without speculative decoding."))
    SPEC_STATE = _nxt

def spec_extra(ctx):
    """GPU memory (MiB) the speculative decoder adds at this window size."""
    if SPEC_STATE == "off":
        return 0
    extra = PROFILE["SPEC_MIB"] + ctx * DRAFT_KV_BYTES / MiB
    if SPEC_STATE == "fast" and DRAFT and os.path.exists(DRAFT):
        extra += os.path.getsize(DRAFT) / MiB
    return extra

# ---------------- chat template ----------------
def prepare_template():
    emb = META.get("tokenizer.chat_template", "") or ""
    note = "the model's own native template (thinking, preserved reasoning and tool calls built in)"
    if not emb:
        note += " - WARNING: the GGUF carries no chat template, llama.cpp will fall back to ChatML"
    return note
TEMPLATE_NOTE = prepare_template()

# ---------------- web tools (stdio MCP server) ----------------
def write_mcp_config(ctx):
    open(WEB_SCRIPT, "w").write(WEB_MCP_SOURCE)
    env = {"WEB_BACKEND": WEB_BACKEND, "WEB_FALLBACK": "1" if AUTO_FALLBACK else "0",
           "PARALLEL_API_KEY": PKEY, "EXA_API_KEY": EKEY, "PYTHONIOENCODING": "utf-8",
           "WEB_MAX_CHARS": str(tool_cap_chars(ctx)), "WEB_DEADLINE": "80"}
    json.dump({"mcpServers": {"web": {"command": sys.executable, "args": ["-u", WEB_SCRIPT],
                                      "env": env, "timeout_ms": 120000}}}, open(MCP_CONFIG, "w"))
    return env

def probe_web(env):
    print(f"\u23f3 Testing web search ({WEB_BACKEND})... ", end="", flush=True)
    try:
        probe = "\n".join(json.dumps(m) for m in [
            {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18"}},
            {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
             "params": {"name": "search", "arguments": {"query": "llama.cpp github"}}}]) + "\n"
        out = subprocess.run([sys.executable, "-u", WEB_SCRIPT], input=probe, capture_output=True, text=True,
                             timeout=120, env=dict(os.environ, **env)).stdout
        res = [json.loads(l) for l in out.splitlines() if l.strip().startswith("{")]
        r2 = next((r for r in res if r.get("id") == 2), {}).get("result", {})
        if r2 and not r2.get("isError"):
            txt = r2["content"][0]["text"]
            print("\u2705" + (f"  (note: {txt.splitlines()[0][1:-1]})" if txt.startswith("[results from") else ""))
        else:
            msg = (r2.get("content") or [{"text": "no reply"}])[0]["text"].splitlines()[0]
            print(f"\u26a0\ufe0f {msg[:300]}\n   The tools stay enabled and retry on every call.")
    except Exception as e:                      # noqa: BLE001
        print(f"\u26a0\ufe0f test failed ({e}); the tools stay enabled and retry on every call.")

# ---------------- model server ----------------
OPTIONAL_FLAGS = {"--fit", "--fit-target", "--cache-ram", "--keep", "--sse-ping-interval", "--cors-origins", "--repeat-penalty",
                  "--min-p", "--reasoning", "--reasoning-budget", "--reasoning-budget-message", "--reasoning-effort",
                  "--reasoning-preserve", "--reasoning-format", "--context-shift", "--ui-config", "--presence-penalty",
                  "--top-k", "--top-p", "--temp", "--image-max-tokens", "--spec-draft-p-min", "--mcp-servers-config",
                  "--ctx-checkpoints", "--timeout"}
DROPPED = set()          # optional flags this llama.cpp build rejected (learned at runtime)
NGL_MODE = "full"        # "full" = every layer on the GPU; "fit" = llama.cpp splits GPU/CPU (a quant too big for the GPU)

class TooBig(RuntimeError):
    """The model file cannot run on this machine at all (never worth retrying with other settings)."""

class Replan(Exception):
    """MTP or vision was switched off while starting: the memory plan was made for a different setup, so plan again
    (the freed memory then goes into a bigger context window)."""

def pkill(name):
    subprocess.run(["pkill", "-x", name], stderr=subprocess.DEVNULL)

def stop(proc):
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill(); proc.wait()

def reply_cap(ctx):
    """Longest single reply the chat UI asks for: automatic = 70% of the window (at most 131,072), so one runaway
    reply can never use the whole window and lock the only slot for hours. -1 = no cap."""
    if MAX_TOKENS is None:
        return min(131072, int(ctx * 0.7))
    return int(MAX_TOKENS)

def cache_ram_mib():
    """Host-RAM prompt cache size: ~12% of system RAM (0.5-4 GB) unless set by hand."""
    if CACHE_RAM_MIB is not None:
        return int(CACHE_RAM_MIB)
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemTotal:"):
                return int(min(4096, max(512, int(line.split()[1]) / 1024 * 0.12)))
    except OSError:
        pass
    return 1024

def server_command(ctx, kv):
    ui_config = json.dumps({"systemMessage": build_system_message(), "agenticMaxTurns": 100, "max_tokens": reply_cap(ctx),
                            "showThoughtInProgress": True})
    cmd = [BIN, "-m", MODEL_PATH, "--host", "127.0.0.1", "--port", str(PORT), "--alias", ALIAS,
           "-c", str(ctx), "--parallel", "1", "-fa", "on", "-ctk", kv, "-ctv", kv, "--jinja",
           "--batch-size", str(BATCH), "--ubatch-size", str(UBATCH)]
    def opt(flag, *vals):
        if flag in DROPPED:
            return
        if flag in OPTIONAL_FLAGS and not has_flag(flag):
            DROPPED.add(flag)
            LOG_NOTES.append(f"this llama.cpp build does not know {flag} - continuing without it")
            return
        cmd.extend([flag] + [str(v) for v in vals])
    if NGL_MODE == "full":
        cmd += ["-ngl", "99"]
        opt("--fit", "off")
    else:                                        # too big for the GPU: let llama.cpp split the layers between GPU and CPU
        opt("--fit", "on")
        opt("--fit-target", 768)
    opt("--cache-ram", cache_ram_mib())
    opt("--ctx-checkpoints", CTX_CHECKPOINTS)
    opt("--timeout", REQUEST_TIMEOUT)
    opt("--sse-ping-interval", 10)
    opt("--cors-origins", "*")                  # MCP tools default CORS to localhost-only; the tunnel/iframe origins differ
    opt("--ui-config", ui_config)
    opt("--temp", TEMP); opt("--top-p", TOP_P); opt("--top-k", TOP_K); opt("--min-p", MIN_P)
    opt("--repeat-penalty", REPEAT_PENALTY)
    if PRESENCE_PENALTY > 0:
        opt("--presence-penalty", PRESENCE_PENALTY)
    if REASONING_KEY == "off":
        opt("--reasoning", "off")
    else:
        opt("--reasoning", "on")
        opt("--reasoning-format", "deepseek")    # reasoning goes to its own panel in the chat UI
        if REASONING_EFFORT:
            opt("--reasoning-effort", REASONING_EFFORT)
        if PRESERVE_THINKING:
            opt("--reasoning-preserve")
        if REASONING_BUDGET > 0:
            opt("--reasoning-budget", REASONING_BUDGET)
            opt("--reasoning-budget-message", " Okay, I have thought about this enough. Time to give the final answer.\n")
    if SPEC_STATE != "off":
        for flag, *vals in spec_flags(SPEC_STATE, MTP_DEPTH[SPEC_STATE], DRAFT):
            if flag == "--spec-draft-p-min":
                opt(flag, *vals)
            else:
                cmd.extend([flag] + [str(v) for v in vals])
    if VISION_STATE != "off":
        cmd += ["--mmproj", MMPROJ]
        if VISION_STATE == "cpu":
            cmd += ["--no-mmproj-offload"]
        opt("--image-max-tokens", IMAGE_MAX_TOKENS)
    if WEB_BACKEND != "off":
        opt("--mcp-servers-config", MCP_CONFIG)
    cmd += ["--path", WEBUI_DIR]
    return cmd

def start_server(ctx, kv):
    if WEB_BACKEND != "off":
        write_mcp_config(ctx)
    return subprocess.Popen(server_command(ctx, kv), stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

def wait_ready(proc, timeout=900):
    t = time.time()
    while time.time() - t < timeout:
        if proc.poll() is not None:
            return False
        try:
            if LOCAL.get(f"http://127.0.0.1:{PORT}/health", timeout=3).status_code == 200:
                time.sleep(1)
                return proc.poll() is None
        except requests.RequestException:
            pass
        time.sleep(2)
    return False

def log_text(max_bytes=2_000_000):
    """The server log (only its last ~2 MB: after days of chatting the file can be huge, and the end is what matters)."""
    try:
        with open(LOG, "rb") as f:
            f.seek(0, os.SEEK_END)
            size = f.tell()
            f.seek(max(0, size - max_bytes))
            return f.read().decode("utf-8", "replace")
    except OSError:
        return ""

def unknown_flag():
    for line in log_text().splitlines():
        l = line.lower()
        if "invalid argument" in l or "error while handling argument" in l or "unknown argument" in l:
            m = re.search(r"(--?[a-z][a-z0-9-]*)", line)
            if m:
                return m.group(1)
    return ""

_ERR_WORDS = ("error", "failed", "unable", "cannot", "not supported", "unsupported", "invalid", "expected", "mismatch")

def fail_reason():
    log = log_text().lower()
    if any(s in log for s in ("out of memory", "cudamalloc failed", "failed to allocate", "unable to allocate",
                              "cuda error: out of memory")):
        return "out of memory"
    if "unknown model architecture" in log:
        return "model architecture not supported by this llama.cpp build"
    if "couldn't bind" in log or "address already in use" in log:
        return "port busy"
    if unknown_flag():
        return "bad argument: " + unknown_flag()
    if "expected 5120, 248320" in log:
        return "speculative: the FastMTP draft needs the patched server build"
    for line in log.splitlines():
        if any(w in line for w in ("speculative", "--spec", "draft", "mtp", "nextn", "d2t")) and any(w in line for w in _ERR_WORDS):
            if any(w in line for w in ("multimodal", "mmproj", "vision", "image")):
                return "speculative: not supported together with vision"
            return "speculative: MTP error"
    for line in log.splitlines():
        if any(w in line for w in ("mmproj", "clip_", "clip:", "mtmd")) and any(w in line for w in ("error", "failed", "unable", "cannot")):
            return "vision (mmproj) error"
    if "flash" in log and ("not supported" in log or "requires" in log):
        return "flash-attention / KV-quantization problem"
    if "failed to load model" in log or "error loading model" in log:
        return "model failed to load"
    if "no space left on device" in log:
        return "out of memory"
    return "see log"

def fully_on_gpu():
    m = re.search(r"offloaded (\d+)/(\d+) layers to gpu", log_text().lower())
    return m is None or m.group(1) == m.group(2)

def mem_label(kv):
    return {"q8_0": "8-bit chat memory", "q4_0": "4-bit chat memory"}.get(kv, "16-bit chat memory")

def spec_label():
    return {"fast": f"FastMTP depth {MTP_DEPTH['fast']}", "embedded": f"MTP depth {MTP_DEPTH['embedded']}"}.get(SPEC_STATE, "off")

def try_launch(ctx, kv):
    """Start the server for one (context, kv) choice, repairing recoverable start-up problems on the way."""
    global PORT, VISION_STATE, SPEC_STATE
    for _ in range(10):
        proc = start_server(ctx, kv)
        if wait_ready(proc):
            return proc, ""
        reason = fail_reason()
        stop(proc)
        if reason.startswith("bad argument"):
            flag = reason.split(": ", 1)[1]
            if flag in OPTIONAL_FLAGS and flag not in DROPPED:
                DROPPED.add(flag)
                print(f"\n   (this llama.cpp build does not know {flag} - continuing without it) ", end="", flush=True)
                continue
            if flag.startswith("--spec") and SPEC_STATE != "off":
                SPEC_STATE = degrade_spec(SPEC_STATE)
                print(f"\n   (this build does not accept {flag} - MTP is now: {spec_label()}) ", end="", flush=True)
                continue
        if reason == "port busy":
            PORT = free_port(PORT + 1)
            print(f"\n   (port was busy, switching to {PORT}) ", end="", flush=True)
            continue
        if reason.startswith("speculative") and SPEC_STATE != "off":
            if "vision" in reason:
                SPEC_STATE = "off"
                print("\n   (this llama.cpp build cannot use MTP together with image input - MTP is OFF, vision stays on) ", end="", flush=True)
            else:
                SPEC_STATE = degrade_spec(SPEC_STATE)
                print(f"\n   ({reason.split(': ', 1)[1]} - MTP is now: {spec_label()}) ", end="", flush=True)
            continue
        if reason == "vision (mmproj) error" and VISION_STATE != "off":
            VISION_STATE = "cpu" if VISION_STATE == "gpu" else "off"
            print(f"\n   (the image encoder failed - continuing with {'the encoder on the CPU' if VISION_STATE == 'cpu' else 'vision OFF, text only'}) ", end="", flush=True)
            continue
        return None, reason
    return None, "too many start-up retries"

def ram_mib():
    """Free system RAM in MiB (0 if unknown)."""
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemAvailable:"):
                return int(line.split()[1]) / 1024
    except OSError:
        pass
    return 0

def gpu_busy():
    """True while the GPU is computing (a very long prompt can keep the server silent for minutes; that is not a hang)."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=15).stdout
        return max(int(x) for x in out.split()) >= 15
    except Exception:                           # noqa: BLE001
        return False

def resources_line():
    used, total = gpu_mem()
    ram = ram_mib()
    try:
        disk = shutil.disk_usage("/content").free / 1e9
    except OSError:
        disk = 0
    return (f"GPU {used}/{total} MiB | RAM free {ram/1024:.1f} GiB | disk free {disk:.0f} GB"
            if used is not None else f"RAM free {ram/1024:.1f} GiB | disk free {disk:.0f} GB")

def launch_best(target):
    """launch_best_once(), but if nothing fits, give back memory: first the image encoder (GPU -> CPU -> off), then MTP."""
    global VISION_STATE, SPEC_STATE
    while True:
        try:
            return launch_best_once(target)
        except Replan:
            continue
        except TooBig:
            raise
        except RuntimeError:
            if VISION_STATE != "off":
                VISION_STATE = "cpu" if VISION_STATE == "gpu" else "off"
                print("   Nothing fit with the image encoder on the GPU - trying again with " +
                      ("the encoder on the CPU." if VISION_STATE == "cpu" else "vision OFF (text only)."))
            elif SPEC_STATE != "off":
                SPEC_STATE = degrade_spec(SPEC_STATE)
                print(f"   Nothing fit with MTP on - trying again with MTP: {spec_label()}.")
            else:
                raise

def launch_partial(target, avail):
    """The file is bigger than the GPU: run it partly on the CPU (llama.cpp's own --fit). Slow, but it runs."""
    global NGL_MODE, VISION_STATE, SPEC_STATE
    gpu_for_weights = max(0, avail - PROFILE["OVERHEAD_MIB"] - 1500 - MIN_FREE_MIB)
    cpu_part = MODEL_MIB - gpu_for_weights
    ram = ram_mib()
    if ram and cpu_part > ram * 0.85:
        raise TooBig(f"This model file ({MODEL_MIB/1024:.1f} GiB) is too large for this machine: about {cpu_part/1024:.1f} GiB would "
                     f"have to sit in system RAM, but only {ram/1024:.1f} GiB is free. In Cell 1 pick a smaller quant "
                     f"(IQ2_M, Q2_K_P and IQ3_XS fit a 15 GB T4), run Cell 1 again, then this cell.")
    NGL_MODE = "fit"
    SPEC_STATE = "off"
    if VISION_STATE == "gpu":
        VISION_STATE = "cpu"
    ctx = max([c for c in PROFILE["CTX_LADDER"] if c <= min(target, 32768)] or [PROFILE["CTX_LADDER"][-1]])
    kv = "q4_0" if KV_PREF == "auto" else KV_PREF
    print(f"   \u26a0\ufe0f This file ({MODEL_MIB/1024:.1f} GiB) is bigger than the {avail/1024:.1f} GiB GPU: about {cpu_part/1024:.1f} GiB will run on the CPU.\n"
          f"      Expect roughly 1-3 tokens/s. The window is limited to {ctx:,} tokens so most layers can stay on the GPU.\n"
          f"      For speed and a long window use IQ2_M / Q2_K_P / IQ3_XS on a T4, or a bigger GPU for this quant.")
    print(f"   trying {ctx:>7,} tokens ({mem_label(kv)}, CPU+GPU split) ... ", end="", flush=True)
    proc, reason = try_launch(ctx, kv)
    if proc is None:
        print(f"\u274c ({reason})")
        print(log_text()[-3000:])
        raise RuntimeError("Could not start the model - see the log above.")
    print("\u2705")
    return proc, ctx, kv

def wait_gpu_released(timeout=40):
    """After a crash or a stop the driver needs a moment to hand the old process's GPU memory back. Planning during that
    moment would needlessly shrink the window, so wait until the memory is free (or the timeout passes)."""
    t0 = time.time()
    used, total = gpu_mem()
    while used is not None and used > 600 and time.time() - t0 < timeout:
        time.sleep(1)
        used, total = gpu_mem()

def launch_best_once(target):
    global NGL_MODE
    wait_gpu_released()
    used0, total = gpu_mem()
    # Subtract at most 1 GiB of "already used" memory: right after a crash the old process may still show up in
    # nvidia-smi for a moment. If something really does hog the GPU, the load fails and the next step is tried.
    avail = (total - min(used0 or 0, 1024)) if total else 15000
    overhead = PROFILE["OVERHEAD_MIB"] + (PROFILE["VISION_MIB"] if VISION_STATE == "gpu" else 0)
    lowest = PROFILE["CTX_LADDER"][-1]
    floor = MODEL_MIB + overhead + MIN_FREE_MIB + kv_mib(lowest, "q4_0", *DIMS) + spec_extra(lowest)
    if total and floor > avail:
        if not ALLOW_CPU_OFFLOAD:
            raise TooBig(f"This model file ({MODEL_MIB/1024:.1f} GiB) is too large for the {avail/1024:.1f} GiB GPU: even a "
                         f"{lowest:,}-token window would not fit. In Cell 1 pick a smaller quant, run Cell 1 again, then this cell.")
        return launch_partial(target, avail)
    NGL_MODE = "full"
    cands = plan_candidates(MODEL_MIB, avail, PROFILE["CTX_LADDER"], target, KV_PREF, DIMS,
                            overhead, MIN_FREE_MIB, auto_order=PROFILE.get("KV_AUTO_ORDER"), extra_fn=spec_extra)
    if not cands:
        cands = [(lowest, "q4_0")]
    print(f"   plan: model {MODEL_MIB/1024:.1f} GiB + chat memory + {overhead} MiB working space"
          f"{' (incl. the image encoder)' if VISION_STATE == 'gpu' else ''}"
          f"{', MTP: ' + spec_label() if SPEC_STATE != 'off' else ''} on a {avail/1024:.1f} GiB GPU")
    state0 = (SPEC_STATE, VISION_STATE)
    adj = 0.0          # MiB by which the estimate proved too optimistic (learned from failed attempts; saves slow model reloads)
    for ctx, kv in cands[:12]:
        need = MODEL_MIB + kv_mib(ctx, kv, *DIMS) + overhead + MIN_FREE_MIB + spec_extra(ctx)
        if adj and need <= avail < need + adj:      # predicted to fit before, but an earlier attempt showed it cannot
            continue
        print(f"   trying {ctx:>7,} tokens ({mem_label(kv)}) ... ", end="", flush=True)
        proc, reason = try_launch(ctx, kv)
        if (SPEC_STATE, VISION_STATE) != state0:       # a feature had to be dropped on the way
            stop(proc)
            print("\u21bb re-planning without it")
            raise Replan()
        if proc is not None and NGL_MODE == "full" and not fully_on_gpu():
            stop(proc); proc, reason = None, "model did not fully fit on the GPU"
        if proc is not None:
            used, tot = gpu_mem()
            if used is not None and tot - used < MIN_FREE_MIB:
                adj = max(adj, (avail - need) + MIN_FREE_MIB - (tot - used) + 30)    # exact error of the estimate
                stop(proc); proc, reason = None, f"fits, but only {tot-used} MiB GPU memory left (unsafe for long chats)"
        if proc is not None:
            print("\u2705")
            return proc, ctx, kv
        if reason == "out of memory":
            adj += 400
        print(f"\u274c ({reason})")
        if reason.startswith("bad argument") or reason in ("model failed to load", "chat template error",
                "model architecture not supported by this llama.cpp build", "vision (mmproj) error") or reason.startswith("speculative"):
            break
        time.sleep(3)
    print(log_text()[-3000:])
    raise RuntimeError("Could not start the model - see the log above.")

# ---------------- vision self-test ----------------
def vision_test():
    """Send a tiny solid-red PNG and ask for its colour. -> (server_ok, answer_text_or_error)."""
    import base64
    b64 = base64.b64encode(make_png_solid(128, 128, (220, 20, 20))).decode()
    try:
        r = LOCAL.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", timeout=300, json={
            "messages": [{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": "data:image/png;base64," + b64}},
                {"type": "text", "text": "What is the main colour of this image? Answer with one word."}]}],
            "max_tokens": 32, "temperature": 0, "chat_template_kwargs": {"enable_thinking": False}})
        if r.status_code != 200:
            return False, f"HTTP {r.status_code}: {r.text[:160]}"
        return True, (r.json()["choices"][0]["message"].get("content") or "").strip()
    except Exception as e:                      # noqa: BLE001
        return False, str(e)[:160]

# ---------------- self tests (non-fatal) ----------------
def smoke_test(ctx):
    base = f"http://127.0.0.1:{PORT}"
    try:
        p = LOCAL.get(base + "/props", timeout=10).json()
        n_ctx = (p.get("default_generation_settings") or {}).get("n_ctx")
        if n_ctx:
            print(f"   server reports a {n_ctx:,}-token window" + ("" if n_ctx >= ctx else f" (asked for {ctx:,})"))
    except Exception as e:                      # noqa: BLE001
        print(f"   (could not read /props: {str(e)[:80]})")
    if WEB_BACKEND != "off":
        try:
            names = [t.get("tool") or t.get("name") for t in LOCAL.get(base + "/tools", timeout=15).json()]
            web = [n for n in names if n and n.startswith("web_")]
            print(("   \u2705 web tools exposed to the model: " + ", ".join(sorted(web))) if web else
                  "   \u26a0\ufe0f the server lists no web_* tools - see /content/llama-server.log")
        except Exception as e:                  # noqa: BLE001
            print(f"   \u26a0\ufe0f could not list tools: {str(e)[:100]}")
    try:
        t0 = time.time()
        r = LOCAL.post(base + "/v1/chat/completions", timeout=180, json={
            "messages": [{"role": "user", "content": "Reply with the single word: ready"}],
            "max_tokens": 24, "temperature": 0, "chat_template_kwargs": {"enable_thinking": False}})
        r.raise_for_status()
        u = r.json().get("usage", {})
        print(f"   \u2705 generation works ({u.get('completion_tokens', '?')} tokens in {time.time()-t0:.1f} s)")
    except Exception as e:                      # noqa: BLE001
        print(f"   \u26a0\ufe0f test generation failed: {str(e)[:160]}")
        return
    if WEB_BACKEND != "off":
        try:
            tools = [{"type": "function", "function": {"name": "web_search", "description": "search the web",
                      "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}]
            r = LOCAL.post(base + "/v1/chat/completions", timeout=180, json={
                "messages": [{"role": "user", "content": "Find the newest llama.cpp release."}],
                "tools": tools, "tool_choice": "required", "max_tokens": 160, "temperature": 0,
                "chat_template_kwargs": {"enable_thinking": False}})
            r.raise_for_status()
            calls = (r.json()["choices"][0]["message"].get("tool_calls") or [])
            print("   \u2705 tool-call parsing works" if calls else
                  "   \u26a0\ufe0f the model answered without a parsed tool call in the self-test (usually still fine in chat)")
        except Exception as e:                  # noqa: BLE001
            print(f"   \u26a0\ufe0f tool-call self-test failed: {str(e)[:200]}")

# ---------------- public link ----------------
def tunnel_ready():
    try:
        return LOCAL.get(f"http://127.0.0.1:{METRICS_PORT}/ready", timeout=3).status_code == 200
    except requests.RequestException:
        return False

def url_answers(url):
    try:
        return requests.get(url + "/health", timeout=8).status_code == 200
    except requests.RequestException:
        return False

def start_cloudflare():
    pkill("cloudflared")
    time.sleep(1)
    for attempt in range(3):
        p = subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                              "--protocol", "http2", "--no-autoupdate",
                              "--metrics", f"127.0.0.1:{METRICS_PORT}"],
                             stdout=open(TLOG, "w"), stderr=subprocess.STDOUT)
        url = None
        for _ in range(60):
            if p.poll() is not None:
                break
            try:
                found = [u for u in re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", open(TLOG).read())
                         if not u.startswith("https://api.")]
            except OSError:
                found = []
            if found:
                url = found[0]
                break
            time.sleep(1)
        if url:
            for _ in range(30):   # wait until the link actually answers
                if p.poll() is not None:
                    break
                if tunnel_ready() and url_answers(url):
                    return p, url
                time.sleep(2)
            if p.poll() is None and tunnel_ready():
                return p, url          # connected; the edge may just need a few more seconds
        stop(p)
        time.sleep(3)
    return None, None

def link_alive(handle):
    return handle is not None and handle.poll() is None and tunnel_ready()

def show_link(url, title):
    display(HTML(f'<div style="font-size:16px;margin:8px 0">{title} <b><a href="{url}" target="_blank">{url}</a></b></div>'))

def server_healthy():
    try:
        return LOCAL.get(f"http://127.0.0.1:{PORT}/health", timeout=8).status_code == 200
    except requests.RequestException:
        return False

# ================= main =================
def main():
    global PORT, METRICS_PORT, VISION_STATE
    pkill("llama-server")
    pkill("cloudflared")
    time.sleep(2)
    PORT, METRICS_PORT = free_port(8090), free_port(20241)
    ensure_webui(cfg)
    web_env = write_mcp_config(TARGET_CONTEXT) if WEB_BACKEND != "off" else {}
    if WEB_BACKEND != "off":
        probe_web(web_env)

    print(f"\u23f3 Loading {PROFILE['NAME']} {QUANT} - target: " +
          ("the largest context that fits" if CONTEXT_WINDOW.strip().lower().startswith("max") else f"{TARGET_CONTEXT:,} tokens") + "...")
    print(f"   chat template: {TEMPLATE_NOTE}")
    print(f"   attention layers with a KV cache: {N_ATTN_LAYER} of {N_LAYER}  |  vision: {VISION_STATE.upper()}  |  MTP: {spec_label()}")
    for n in LOG_NOTES:
        print(f"   \u2139\ufe0f {n}")
    del LOG_NOTES[:]
    server, CTX, KV = launch_best(TARGET_CONTEXT)

    used, total = gpu_mem()
    think_txt = ("OFF" if REASONING_KEY == "off" else "MAX (xhigh effort, unlimited)" if REASONING_BUDGET < 0
                 else f"{REASONING.split()[0]} ({REASONING_EFFORT} effort, up to {REASONING_BUDGET:,} tokens)")
    print(f"\n\u2705 Model ready  |  model: {QUANT}  |  context: {CTX:,} tokens ({mem_label(KV)})  |  reasoning: {think_txt}")
    print(f"   speed-up: {'MTP ' + spec_label() if SPEC_STATE != 'off' else 'none (maximum context)'}  |  vision: {VISION_STATE.upper()}"
          + ("  |  CPU+GPU split (slow)" if NGL_MODE == "fit" else ""))
    for n in LOG_NOTES:
        print(f"   \u2139\ufe0f {n}")
    del LOG_NOTES[:]
    if "Max" not in CONTEXT_WINDOW and CTX < TARGET_CONTEXT:
        print(f"   \u2139\ufe0f {TARGET_CONTEXT:,} didn't fit in GPU memory; using {CTX:,}.")
    if "Max" in CONTEXT_WINDOW and CTX < PROFILE["CTX_LADDER"][0]:
        print(f"   \u2139\ufe0f {CTX:,} is the most this GPU can hold with this model file"
              + (" (MTP / vision use some of that memory; switch them off for a bigger window)." if (SPEC_STATE != "off" or VISION_STATE == "gpu") else "."))
    web_txt = "OFF"
    if WEB_BACKEND != "off":
        web_txt = "ON \u00b7 " + WEB_BACKEND + (" + auto-fallback" if AUTO_FALLBACK else "")
    cap_txt = f"  |  one web result is capped at {tool_cap_chars(CTX):,} characters" if WEB_BACKEND != "off" else ""
    print(f"   web search: {web_txt}{cap_txt}  |  GPU memory: {used} / {total} MiB")
    while VISION_STATE != "off":
        print(f"\u23f3 Testing image input (vision on the {'GPU' if VISION_STATE == 'gpu' else 'CPU'})... ", end="", flush=True)
        ok, ans = vision_test()
        if ok:
            print(f"\u2705  the model sees images (test answer: \u201c{ans[:40]}\u201d)" +
                  ("" if "red" in ans.lower() else "  (unexpected answer - check with a real image)"))
            break
        print(f"\u26a0\ufe0f failed ({ans})")
        stop(server)
        VISION_STATE = "cpu" if VISION_STATE == "gpu" else "off"
        print("   Restarting with " + ("the encoder on the CPU (slower image encoding)..." if VISION_STATE == "cpu" else "vision OFF (text only)..."))
        server, CTX, KV = launch_best(TARGET_CONTEXT)
    if VISION_PREF != "off" and VISION_STATE == "off":
        print("   \u2139\ufe0f Running text-only: image attachments will not work this session.")
    smoke_test(CTX)
    print()

    tunnel, public_url = None, None
    try:
        from google.colab import output
        # Colab's private new-tab links (*.prod.colab.dev) return 404 in modern browsers,
        # so only the in-notebook view is shown; use the public link for a full tab.
        if SHOW_INLINE:
            print("\U0001F4AC Chat inside the notebook (private, only you):")
            output.serve_kernel_port_as_iframe(PORT, height=850)
        if LINK_MODE != "off":
            print(f"\u23f3 Opening the public link ({LINK_MODE})...")
            tunnel, public_url = start_cloudflare()
            if public_url:
                show_link(public_url, "\U0001F30D Public link:")
                print("   (If it says it can't be reached, wait 30 s and refresh, or open it in an incognito window.)")
            else:
                print("\u26a0\ufe0f The public link didn't start yet - it keeps retrying below. The chat above works meanwhile.")

        print("\nHow to use: open a link \u2192 type your message. Long text/code (1,000+ lines) is fine: paste it or attach the file.")
        if WEB_BACKEND != "off":
            print("Web search: use the \u201cAllow tool calls: ON / OFF\u201d switch inside the chat box, right after the + button, to control whether searches ask for permission (applies instantly, no reload).")
        print("Keep this cell running. Press \u25a0 (stop) to shut everything down.\n")

        start, last_report = time.time(), 0
        server_restarts = tunnel_restarts = bad_ticks = bad_health = unhealthy_total = 0
        last_restart, tunnel_port = 0.0, PORT
        while True:
            time.sleep(MONITOR_SECONDS)

            # restarts that are long ago no longer count: a session that runs for days must not run out of them
            if server_restarts and time.time() - last_restart > FORGET_RESTARTS_MIN * 60:
                server_restarts = 0

            # 1) model server: restart on crash (smaller context if it ran out of memory) or if it really stopped answering
            crashed = server.poll() is not None
            if not crashed:
                if server_healthy():
                    bad_health = unhealthy_total = 0
                else:
                    bad_health += 1
                    unhealthy_total += 1
            # ~2 minutes without an answer = hung, UNLESS the GPU is busy (a very long prompt keeps the server silent for
            # minutes; killing it would throw the work away). Even then, 15 minutes of silence is a hang.
            hung = (bad_health >= 6 and not gpu_busy()) or unhealthy_total >= 45
            if crashed or hung:
                if server_restarts >= MAX_SERVER_RESTARTS:
                    print("\u274c Model server keeps failing. Last log lines:\n")
                    print(log_text()[-3000:])
                    break
                server_restarts += 1
                last_restart = time.time()
                reason = "hung (no answer for a long time)" if hung else fail_reason()
                stop(server)
                target = CTX
                if reason == "out of memory":
                    smaller = [c for c in PROFILE["CTX_LADDER"] if c < CTX]
                    if smaller:
                        target = smaller[0]
                        print(f"\u26a0\ufe0f Ran out of GPU memory at {CTX:,} tokens - restarting with at most {target:,}.")
                print(f"\u26a0\ufe0f Model server stopped ({reason}) - restarting ({server_restarts}/{MAX_SERVER_RESTARTS})...")
                try:
                    server, CTX, KV = launch_best(target)
                    bad_health = unhealthy_total = 0
                    print("\u2705 Model server is back. Your chats are kept in your browser; just resend the last message.")
                except RuntimeError as e:
                    print(f"\u274c {e}")
                continue

            # 2) public link (also re-pointed if a restart had to move the server to another port)
            if LINK_MODE != "off":
                bad_ticks = 0 if link_alive(tunnel) else bad_ticks + 1
                if tunnel is None or bad_ticks >= 3 or tunnel_port != PORT:
                    tunnel_restarts += 1
                    print(f"\u26a0\ufe0f Public link dropped - reconnecting (#{tunnel_restarts})...")
                    stop(tunnel)
                    tunnel, new_url = start_cloudflare()
                    tunnel_port = PORT
                    bad_ticks = 0
                    if new_url and new_url != public_url:
                        public_url = new_url
                        show_link(public_url, "\U0001F30D NEW public link (the old one no longer works):")
                    elif new_url:
                        print("\u2705 Public link reconnected (same address).")

            # 3) heartbeat every 10 min, with a look at the machine's memory
            mins = int((time.time() - start) // 60)
            if mins - last_report >= 10:
                last_report = mins
                print(f"\u23f1 Up {mins} min | server restarts: {server_restarts} | link restarts: {tunnel_restarts} | {resources_line()}")
                if 0 < ram_mib() < 700:
                    print("\u26a0\ufe0f System RAM is nearly full (saved prompt snapshots of long chats). If the server is ever stopped by the system "
                          "it restarts by itself; a new chat frees the most.")
    except KeyboardInterrupt:
        print("Stopping...")
    finally:
        stop(tunnel)
        stop(server)
        print("Shut down.")

if __name__ == "__main__":
    main()


## Troubleshooting

- **How much context did I get?** Cell 2 prints it (“context: … tokens”) and why it is below 262,144. On a 15 GB T4 that is normal: the weights alone take 10 GB. For the maximum keep **vision Off** and **speed-up Off**; a smaller quant (Q2_K_P is close in size) does not give more room than IQ2_M.
- **It is slow (a few tokens per second)** → a 27B model on a T4 is memory-bound. Turn on **Embedded MTP** (or build with **FastMTP** in Cell 1 and pick it in Cell 2) for roughly 1.6–3× — at the price of context. *Reasoning = Max* also writes a lot of thinking text; **High / Medium** shorten it.
- **“MTP is not available / MTP is OFF”** → Cell 2 told you why (this build lacks MTP flags, the FastMTP patch is not in the server, or the build cannot combine MTP with images). It already carries on without it. For FastMTP choose the FastMTP option in Cell 1 and run both cells again.
- **“the FastMTP patch could not be applied”** (Cell 1) → the notebook tried the verified release, then the model author's exact llama.cpp commit, then fell back to the standard build. Embedded MTP still works. Details are in `/content/build.log`.
- **“the image encoder failed” / “vision on the CPU”** → Cell 2 handled it. CPU image encoding works but takes tens of seconds per image. Smaller `IMAGE_MAX_TOKENS` (advanced block) frees GPU memory.
- **Images are ignored** → vision is **Off by default** in Cell 2; pick one of the “On” modes, and make sure Cell 1 printed “Vision: mmproj-…” (tick *DOWNLOAD_VISION*, re-run Cell 1 if not).
- **The model thinks for a very long time / seems to loop** → that is *Max* reasoning (heavily quantized models can loop). Press stop, or pick **High** or **Medium**. A reply is capped automatically, so it can never block the chat for good. If it repeats itself, set `PRESENCE_PENALTY = 1.5` in the advanced block.
- **“context full”** → start a new chat. To make chats last longer, set `PRESERVE_THINKING = False` in the advanced block (earlier turns' reasoning is then not kept; follow-up turns re-read a bit more).
- **First answer after a long pause is slow / the page says “thinking” for minutes** → the server is reading a long prompt (or a new chat after another one). Leave it; it is not frozen and the watchdog will not restart a busy GPU.
- **“Model server stopped … restarting”** → the watchdog is repairing it (a crash or a real hang). Wait for “Model server is back”, then resend your last message. Your chats are kept in the browser.
- **“System RAM is nearly full”** → many long chats' saved snapshots. Start a new chat; if the system ever stops the server it restarts by itself.
- **Web search says rate-limited / failed** → it falls back automatically (Parallel → DuckDuckGo → Exa). For heavy use, add a free key from platform.parallel.ai as a secret named `PARALLEL_API_KEY`.
- **Tool prompts keep appearing / never appear** → use the **Allow tool calls: ON / OFF** switch inside the chat box. If it is missing, hard-refresh once (Ctrl+Shift+R / Cmd+Shift+R). If the build lacks MCP support Cell 2 prints a note and the model runs without web tools.
- **“this llama.cpp build does not know --xyz”** → harmless: optional flags a build does not understand are dropped and the notebook carries on.
- **Public link doesn't open** → wait ~30 s and refresh. The link changes every run; the watchdog reconnects it and prints the new address if it drops.
- **“q8_0 / 4-bit chat memory”** → that is the conversation cache (KV cache), not the model. The model file is shown as `model:`.
- **Colab disconnects** → free Colab ends idle or long sessions (usually after a few hours; Google decides, and this notebook does not try to get around it). Re-run Cell 1 (fast: everything finished is reused, and the Drive cache skips the compile) and then Cell 2.
- **Download or build failed** → just run Cell 1 again; finished parts are reused.
- **Logs** → `/content/build.log`, `/content/llama-server.log`, `/content/tunnel.log`.